In [1]:
print("Hello World")

Hello World


In [3]:
import requests, re, pandas as pd, time
print("Success!")


Success!


In [8]:
import requests
import re
import pandas as pd
import time
import random
from datetime import datetime

# 1. Configuration
TARGET_SUBREDDITS = ['technology', 'science', 'news', 'gaming', 'politics']
YEARS = range(2018, 2026)
OUTPUT_FILE = "reddit_network_2018_2025.csv"

def extract_links(post):
    text = f"{post.get('title', '')} {post.get('selftext', '')}"
    return list(set(re.findall(r'r/([A-Za-z0-9_]+)', text)))

all_edges = []

for sub in TARGET_SUBREDDITS:
    for year in YEARS:
        # Convert YYYY-MM-DD to Unix Timestamp (Integer)
        after_ts = int(datetime(year, 1, 1).timestamp())
        before_ts = int(datetime(year, 12, 31).timestamp())
        
        print(f"Requesting {sub} ({year})...", end=" ")
        
        url = "https://api.pullpush.io"
        params = {
            'subreddit': sub,
            'after': after_ts,
            'before': before_ts,
            'size': 100
        }
        
        try:
            r = requests.get(url, params=params, timeout=15)
            
            if r.status_code == 200:
                data = r.json().get('data', [])
                print(f"Success! Found {len(data)} posts.")
                for post in data:
                    targets = extract_links(post)
                    for t in targets:
                        if t.lower() != sub.lower():
                            all_edges.append({
                                'SOURCE': sub, 'TARGET': t, 
                                'TIMESTAMP': post.get('created_utc'), 'POST_ID': post.get('id')
                            })
                # Add a longer, random delay to stay under the radar
                time.sleep(random.uniform(3.0, 6.0)) 
                
            elif r.status_code == 429:
                print("Rate Limited. Sleeping 60s...")
                time.sleep(60)
            else:
                print(f"Failed with Status {r.status_code}: {r.text[:50]}")
                time.sleep(5)
                
        except Exception as e:
            print(f"Connection Error: {e}")
            time.sleep(10)

# 2. Save
if all_edges:
    pd.DataFrame(all_edges).to_csv(OUTPUT_FILE, index=False)
    print(f"\nSaved {len(all_edges)} total edges to {OUTPUT_FILE}")


Requesting technology (2018)... Success! Found 0 posts.
Requesting technology (2019)... Success! Found 0 posts.
Requesting technology (2020)... Success! Found 0 posts.
Requesting technology (2021)... Success! Found 0 posts.
Requesting technology (2022)... Success! Found 0 posts.
Requesting technology (2023)... Success! Found 0 posts.
Requesting technology (2024)... Success! Found 0 posts.
Requesting technology (2025)... Success! Found 0 posts.
Requesting science (2018)... Success! Found 0 posts.
Requesting science (2019)... Success! Found 0 posts.
Requesting science (2020)... Success! Found 0 posts.
Requesting science (2021)... Success! Found 0 posts.
Requesting science (2022)... Success! Found 0 posts.
Requesting science (2023)... Success! Found 0 posts.
Requesting science (2024)... Success! Found 0 posts.
Requesting science (2025)... Success! Found 0 posts.
Requesting news (2018)... Success! Found 0 posts.
Requesting news (2019)... Success! Found 0 posts.
Requesting news (2020)... Su

# Political Submission

In [15]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from datetime import datetime
import os

In [16]:

file_path = "subreddits24/politics_submissions.zst"

def find_exact_range(path):
    with open(path, 'rb') as fh:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8')
            
            first_post = True
            last_timestamp = None
            count = 0
            
            for line in text_stream:
                try:
                    post = json.loads(line)
                    ts = int(post['created_utc'])
                    
                    if first_post:
                        print(f"START DATE: {datetime.fromtimestamp(ts)}")
                        first_post = False
                    
                    last_timestamp = ts
                    count += 1
                    
                    if count % 1000000 == 0:
                        print(f"Still scanning... current date in file: {datetime.fromtimestamp(ts)}")
                        
                except:
                    continue
            
            if last_timestamp:
                print(f"\nFINAL END DATE: {datetime.fromtimestamp(last_timestamp)}")

find_exact_range(file_path)


START DATE: 2007-08-06 11:30:51
Still scanning... current date in file: 2011-11-16 06:46:14
Still scanning... current date in file: 2015-11-19 02:21:39
Still scanning... current date in file: 2018-03-31 16:42:29
Still scanning... current date in file: 2020-10-01 21:32:08

FINAL END DATE: 2025-01-01 05:58:24


# Space-Saving "Streaming" Strategy
only keeping the tiny hyperlinks

## politics subreddit

In [2]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/politics_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'politics')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("politics_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 0
Lines: 1600000 | Edges Found: 0
Lines: 1700000 | Edges Found: 0
Lines: 1800000 | Edges Found: 163
Lines: 1900000 | Edges Found: 487
Lines: 2000000 | Edges Found: 769
Lines: 2100000 | Edges Found: 1101
Lines: 2200000 | Edges Found: 1450
Lines: 2300000 | Edges Found: 1673
Lines: 2400000 | Edges Found: 2020
Lines: 2500000 | Edges Found: 2137
Lines: 2600000 | Edges Found: 2270
Lines: 2700000 | Edges Found: 2362
Lines: 2800000 | Edges Found: 2447
Lines: 2900000 | Edges Found: 2521
Lines: 3000000 | Edges Found: 2622
Lines: 310000

## science subreddit

In [3]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/science_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'science')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("science_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 179
Lines: 600000 | Edges Found: 620
Lines: 700000 | Edges Found: 891
Lines: 800000 | Edges Found: 1068
Lines: 900000 | Edges Found: 1218
Complete! Extracted 1240 edges.


##  technology subreddit

In [4]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/technology_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'technology')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("technology_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 111
Lines: 1100000 | Edges Found: 664
Lines: 1200000 | Edges Found: 1290
Lines: 1300000 | Edges Found: 1666
Lines: 1400000 | Edges Found: 1993
Lines: 1500000 | Edges Found: 2259
Lines: 1600000 | Edges Found: 2552
Lines: 1700000 | Edges Found: 2715
Lines: 1800000 | Edges Found: 2935
Lines: 1900000 | Edges Found: 3081
Lines: 2000000 | Edges Found: 3184
Lines: 2100000 | Edges Found: 3372
Lines: 2200000 | Edges Found: 3642
Complete! Extracted 3648 edges.


# dankmemes subreddit

In [5]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/dankmemes_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'dankmemes')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("dankmemes_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 465
Lines: 200000 | Edges Found: 930
Lines: 300000 | Edges Found: 1335
Lines: 400000 | Edges Found: 1678
Lines: 500000 | Edges Found: 2081
Lines: 600000 | Edges Found: 2528
Lines: 700000 | Edges Found: 2839
Lines: 800000 | Edges Found: 3186
Lines: 900000 | Edges Found: 3608
Lines: 1000000 | Edges Found: 4237
Lines: 1100000 | Edges Found: 4672
Lines: 1200000 | Edges Found: 5016
Lines: 1300000 | Edges Found: 5352
Lines: 1400000 | Edges Found: 5636
Lines: 1500000 | Edges Found: 5966
Lines: 1600000 | Edges Found: 6287
Lines: 1700000 | Edges Found: 6529
Lines: 1800000 | Edges Found: 6785
Lines: 1900000 | Edges Found: 7019
Lines: 2000000 | Edges Found: 7211
Lines: 2100000 | Edges Found: 7455
Lines: 2200000 | Edges Found: 7774
Lines: 2300000 | Edges Found: 8073
Lines: 2400000 | Edges Found: 8376
Lines: 2500000 | Edges Found: 8674
Lines: 2600000 | Edges Found: 8934
Lines: 2700000 | Edges Found: 9215
Lines: 2800000 | Edges Found: 9470
Lines: 2900000 | Edges Found: 9

# gaming subreddit

In [6]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/gaming_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'gaming')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("gaming_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 0
Lines: 1600000 | Edges Found: 0
Lines: 1700000 | Edges Found: 0
Lines: 1800000 | Edges Found: 0
Lines: 1900000 | Edges Found: 106
Lines: 2000000 | Edges Found: 1799
Lines: 2100000 | Edges Found: 3540
Lines: 2200000 | Edges Found: 5065
Lines: 2300000 | Edges Found: 6623
Lines: 2400000 | Edges Found: 8173
Lines: 2500000 | Edges Found: 9942
Lines: 2600000 | Edges Found: 11556
Lines: 2700000 | Edges Found: 12873
Lines: 2800000 | Edges Found: 14048
Lines: 2900000 | Edges Found: 15166
Lines: 3000000 | Edges Found: 16206
Lines: 31

# pcmasterrace subreddit

In [7]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/pcmasterrace_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'pcmasterrace')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("pcmasterrace_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2503
Lines: 200000 | Edges Found: 5380
Lines: 300000 | Edges Found: 8471
Lines: 400000 | Edges Found: 11536
Lines: 500000 | Edges Found: 14853
Lines: 600000 | Edges Found: 17776
Lines: 700000 | Edges Found: 20543
Lines: 800000 | Edges Found: 25059
Lines: 900000 | Edges Found: 29358
Lines: 1000000 | Edges Found: 33862
Lines: 1100000 | Edges Found: 37796
Lines: 1200000 | Edges Found: 40950
Lines: 1300000 | Edges Found: 43792
Lines: 1400000 | Edges Found: 46125
Lines: 1500000 | Edges Found: 48293
Lines: 1600000 | Edges Found: 50397
Lines: 1700000 | Edges Found: 52544
Lines: 1800000 | Edges Found: 54541
Lines: 1900000 | Edges Found: 56293
Lines: 2000000 | Edges Found: 57955
Lines: 2100000 | Edges Found: 59315
Lines: 2200000 | Edges Found: 60591
Lines: 2300000 | Edges Found: 61707
Lines: 2400000 | Edges Found: 62771
Lines: 2500000 | Edges Found: 64065
Lines: 2600000 | Edges Found: 65411
Lines: 2700000 | Edges Found: 66680
Lines: 2800000 | Edges Found: 68215
Line

# worldnews_submissions subreddit

In [8]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/worldnews_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'worldnews')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("worldnews_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 340
Lines: 1300000 | Edges Found: 620
Lines: 1400000 | Edges Found: 761
Lines: 1500000 | Edges Found: 850
Lines: 1600000 | Edges Found: 960
Lines: 1700000 | Edges Found: 1046
Lines: 1800000 | Edges Found: 1098
Lines: 1900000 | Edges Found: 1205
Lines: 2000000 | Edges Found: 1263
Lines: 2100000 | Edges Found: 1318
Lines: 2200000 | Edges Found: 1384
Lines: 2300000 | Edges Found: 1444
Lines: 2400000 | Edges Found: 1489
Lines: 2500000 | Edges Found: 1543
Lines: 2600000 | Edges Found: 1641
Lines: 2700000 | Edges Found: 1692
Lines: 2800000 | Edges Found: 1782
Lines: 2900000 | Edges Found: 2058
Lines: 3000000 | Edges Found: 21

# 3Dprinting_submissions subreddit

In [9]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/3Dprinting_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', '3Dprinting')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("3Dprinting_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2752
Lines: 200000 | Edges Found: 4530
Lines: 300000 | Edges Found: 5879
Lines: 400000 | Edges Found: 7021
Lines: 500000 | Edges Found: 8048
Lines: 600000 | Edges Found: 9202
Lines: 700000 | Edges Found: 10658
Complete! Extracted 12096 edges.


# AdviceAnimals_submissions subreddit

In [10]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/AdviceAnimals_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'AdviceAnimals')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("AdviceAnimals_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 0
Lines: 1600000 | Edges Found: 0
Lines: 1700000 | Edges Found: 0
Lines: 1800000 | Edges Found: 0
Lines: 1900000 | Edges Found: 0
Lines: 2000000 | Edges Found: 0
Lines: 2100000 | Edges Found: 0
Lines: 2200000 | Edges Found: 0
Lines: 2300000 | Edges Found: 0
Lines: 2400000 | Edges Found: 0
Lines: 2500000 | Edges Found: 0
Lines: 2600000 | Edges Found: 246
Lines: 2700000 | Edges Found: 1094
Lines: 2800000 | Edges Found: 1984
Lines: 2900000 | Edges Found: 2999
Lines: 3000000 | Edges Found: 3745
Lines: 3100000 | Edges Found: 4647


# AskDocs_submissions subreddit

In [11]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/AskDocs_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'AskDocs')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("AskDocs_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2981
Lines: 200000 | Edges Found: 5002
Lines: 300000 | Edges Found: 7036
Lines: 400000 | Edges Found: 8927
Lines: 500000 | Edges Found: 10423
Lines: 600000 | Edges Found: 11945
Lines: 700000 | Edges Found: 13275
Lines: 800000 | Edges Found: 14812
Lines: 900000 | Edges Found: 16124
Lines: 1000000 | Edges Found: 17224
Lines: 1100000 | Edges Found: 18630
Lines: 1200000 | Edges Found: 20194
Lines: 1300000 | Edges Found: 21588
Lines: 1400000 | Edges Found: 23045
Lines: 1500000 | Edges Found: 25073
Lines: 1600000 | Edges Found: 27042
Lines: 1700000 | Edges Found: 28943
Lines: 1800000 | Edges Found: 30702
Complete! Extracted 32303 edges.


# CryptoCurrency_submissions subreddit

In [12]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/CryptoCurrency_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'CryptoCurrency')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("CryptoCurrency_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3358
Lines: 200000 | Edges Found: 6037
Lines: 300000 | Edges Found: 8289
Lines: 400000 | Edges Found: 10379
Lines: 500000 | Edges Found: 12395
Lines: 600000 | Edges Found: 13588
Lines: 700000 | Edges Found: 14485
Lines: 800000 | Edges Found: 16057
Lines: 900000 | Edges Found: 18100
Lines: 1000000 | Edges Found: 19378
Lines: 1100000 | Edges Found: 21403
Lines: 1200000 | Edges Found: 23179
Lines: 1300000 | Edges Found: 24685
Lines: 1400000 | Edges Found: 26219
Lines: 1500000 | Edges Found: 28029
Lines: 1600000 | Edges Found: 30106
Lines: 1700000 | Edges Found: 32181
Lines: 1800000 | Edges Found: 34909
Lines: 1900000 | Edges Found: 37614
Complete! Extracted 38750 edges.


# DestinyTheGame_submissions subreddit

In [13]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/DestinyTheGame_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'DestinyTheGame')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("DestinyTheGame_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2284
Lines: 200000 | Edges Found: 5289
Lines: 300000 | Edges Found: 8571
Lines: 400000 | Edges Found: 12204
Lines: 500000 | Edges Found: 14572
Lines: 600000 | Edges Found: 18682
Lines: 700000 | Edges Found: 22093
Lines: 800000 | Edges Found: 25668
Lines: 900000 | Edges Found: 28115
Lines: 1000000 | Edges Found: 30702
Lines: 1100000 | Edges Found: 32616
Lines: 1200000 | Edges Found: 34877
Lines: 1300000 | Edges Found: 37092
Lines: 1400000 | Edges Found: 38870
Lines: 1500000 | Edges Found: 41020
Lines: 1600000 | Edges Found: 42864
Lines: 1700000 | Edges Found: 44413
Lines: 1800000 | Edges Found: 46211
Lines: 1900000 | Edges Found: 47992
Lines: 2000000 | Edges Found: 49800
Lines: 2100000 | Edges Found: 51255
Lines: 2200000 | Edges Found: 54036
Complete! Extracted 54642 edges.


# GunAccessoriesForSale_submissions subreddit

In [14]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/GunAccessoriesForSale_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'GunAccessoriesForSale')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("GunAccessoriesForSale_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2024
Lines: 200000 | Edges Found: 3674
Lines: 300000 | Edges Found: 5486
Lines: 400000 | Edges Found: 7552
Lines: 500000 | Edges Found: 9622
Lines: 600000 | Edges Found: 11718
Lines: 700000 | Edges Found: 13526
Lines: 800000 | Edges Found: 15507
Lines: 900000 | Edges Found: 18021
Lines: 1000000 | Edges Found: 20142
Lines: 1100000 | Edges Found: 22270
Lines: 1200000 | Edges Found: 24498
Complete! Extracted 24653 edges.


# hiphopheads_submissions subreddit

In [15]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/hiphopheads_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'hiphopheads')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("hiphopheads_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 728
Lines: 300000 | Edges Found: 3508
Lines: 400000 | Edges Found: 5480
Lines: 500000 | Edges Found: 7086
Lines: 600000 | Edges Found: 8640
Lines: 700000 | Edges Found: 10175
Lines: 800000 | Edges Found: 11364
Lines: 900000 | Edges Found: 12382
Lines: 1000000 | Edges Found: 13554
Lines: 1100000 | Edges Found: 14673
Lines: 1200000 | Edges Found: 15804
Lines: 1300000 | Edges Found: 17363
Lines: 1400000 | Edges Found: 18203
Lines: 1500000 | Edges Found: 19262
Complete! Extracted 19752 edges.


# nba_submissions subreddit

In [16]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/nba_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'nba')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("nba_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 4367
Lines: 300000 | Edges Found: 12109
Lines: 400000 | Edges Found: 16976
Lines: 500000 | Edges Found: 22845
Lines: 600000 | Edges Found: 27791
Lines: 700000 | Edges Found: 30069
Lines: 800000 | Edges Found: 32949
Lines: 900000 | Edges Found: 35778
Lines: 1000000 | Edges Found: 37652
Lines: 1100000 | Edges Found: 43676
Lines: 1200000 | Edges Found: 48667
Lines: 1300000 | Edges Found: 50922
Lines: 1400000 | Edges Found: 57298
Lines: 1500000 | Edges Found: 61011
Lines: 1600000 | Edges Found: 64351
Lines: 1700000 | Edges Found: 72647
Lines: 1800000 | Edges Found: 80031
Lines: 1900000 | Edges Found: 85215
Lines: 2000000 | Edges Found: 91677
Lines: 2100000 | Edges Found: 98998
Complete! Extracted 102718 edges.


# soccer_submissions subreddit

In [20]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/soccer_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'soccer')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("soccer_submissions_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 77
Lines: 300000 | Edges Found: 4403
Lines: 400000 | Edges Found: 8729
Lines: 500000 | Edges Found: 12111
Lines: 600000 | Edges Found: 17191
Lines: 700000 | Edges Found: 21376
Lines: 800000 | Edges Found: 23070
Lines: 900000 | Edges Found: 25101
Lines: 1000000 | Edges Found: 27900
Lines: 1100000 | Edges Found: 31460
Lines: 1200000 | Edges Found: 34089
Lines: 1300000 | Edges Found: 36995
Lines: 1400000 | Edges Found: 39663
Lines: 1500000 | Edges Found: 44338
Lines: 1600000 | Edges Found: 49150
Lines: 1700000 | Edges Found: 53127
Lines: 1800000 | Edges Found: 58834
Complete! Extracted 62328 edges.


# technology_submissions subreddit

In [18]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/technology_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'technology')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("technology_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 111
Lines: 1100000 | Edges Found: 664
Lines: 1200000 | Edges Found: 1290
Lines: 1300000 | Edges Found: 1666
Lines: 1400000 | Edges Found: 1993
Lines: 1500000 | Edges Found: 2259
Lines: 1600000 | Edges Found: 2552
Lines: 1700000 | Edges Found: 2715
Lines: 1800000 | Edges Found: 2935
Lines: 1900000 | Edges Found: 3081
Lines: 2000000 | Edges Found: 3184
Lines: 2100000 | Edges Found: 3372
Lines: 2200000 | Edges Found: 3642
Complete! Extracted 3648 edges.


# unpopularopinion_submissions subreddit

In [19]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/unpopularopinion_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'unpopularopinion')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("unpopularopinion_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 4492
Lines: 200000 | Edges Found: 10207
Lines: 300000 | Edges Found: 16087
Lines: 400000 | Edges Found: 22465
Lines: 500000 | Edges Found: 27406
Lines: 600000 | Edges Found: 31638
Lines: 700000 | Edges Found: 35520
Lines: 800000 | Edges Found: 39086
Lines: 900000 | Edges Found: 42186
Lines: 1000000 | Edges Found: 44442
Lines: 1100000 | Edges Found: 46607
Lines: 1200000 | Edges Found: 48848
Lines: 1300000 | Edges Found: 50867
Lines: 1400000 | Edges Found: 52658
Lines: 1500000 | Edges Found: 55017
Lines: 1600000 | Edges Found: 57196
Lines: 1700000 | Edges Found: 58902
Lines: 1800000 | Edges Found: 60246
Lines: 1900000 | Edges Found: 61692
Lines: 2000000 | Edges Found: 62916
Lines: 2100000 | Edges Found: 63934
Lines: 2200000 | Edges Found: 65154
Lines: 2300000 | Edges Found: 66135
Complete! Extracted 67151 edges.


# Warhammer40k_submissions subreddit

In [21]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Warhammer40k_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Warhammer40k')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Warhammer40k_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2568
Lines: 200000 | Edges Found: 3973
Lines: 300000 | Edges Found: 4903
Lines: 400000 | Edges Found: 5786
Lines: 500000 | Edges Found: 6481
Lines: 600000 | Edges Found: 7203
Lines: 700000 | Edges Found: 8086
Lines: 800000 | Edges Found: 9057
Complete! Extracted 9122 edges.


# Watchexchange_submissions subreddit

In [22]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Watchexchange_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Watchexchange')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Watchexchange_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2564
Lines: 200000 | Edges Found: 2709
Lines: 300000 | Edges Found: 2850
Lines: 400000 | Edges Found: 3003
Lines: 500000 | Edges Found: 3093
Complete! Extracted 3163 edges.


# atheism_submissions subreddit

In [23]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/atheism_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'atheism')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("atheism_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 622
Lines: 900000 | Edges Found: 4258
Lines: 1000000 | Edges Found: 7455
Lines: 1100000 | Edges Found: 9960
Lines: 1200000 | Edges Found: 11813
Lines: 1300000 | Edges Found: 13645
Complete! Extracted 14145 edges.


# Bitcoin_submissions subreddit

In [24]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Bitcoin_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Bitcoin')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Bitcoin_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 2441
Lines: 300000 | Edges Found: 5530
Lines: 400000 | Edges Found: 7913
Lines: 500000 | Edges Found: 10315
Lines: 600000 | Edges Found: 11967
Lines: 700000 | Edges Found: 13532
Lines: 800000 | Edges Found: 15399
Lines: 900000 | Edges Found: 17041
Lines: 1000000 | Edges Found: 18322
Lines: 1100000 | Edges Found: 19631
Lines: 1200000 | Edges Found: 21115
Complete! Extracted 22389 edges.


# blender_submissions subreddit

In [25]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/blender_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'blender')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("blender_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1402
Lines: 200000 | Edges Found: 2176
Lines: 300000 | Edges Found: 2776
Lines: 400000 | Edges Found: 3454
Lines: 500000 | Edges Found: 4048
Lines: 600000 | Edges Found: 5120
Complete! Extracted 5306 edges.


# business_submissions subreddit

In [26]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/business_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'business')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("business_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 29
Lines: 600000 | Edges Found: 226
Lines: 700000 | Edges Found: 575
Lines: 800000 | Edges Found: 688
Lines: 900000 | Edges Found: 759
Lines: 1000000 | Edges Found: 830
Lines: 1100000 | Edges Found: 900
Lines: 1200000 | Edges Found: 1020
Lines: 1300000 | Edges Found: 1080
Lines: 1400000 | Edges Found: 1132
Lines: 1500000 | Edges Found: 1203
Lines: 1600000 | Edges Found: 1314
Lines: 1700000 | Edges Found: 1437
Lines: 1800000 | Edges Found: 1562
Lines: 1900000 | Edges Found: 1721
Lines: 2000000 | Edges Found: 1837
Lines: 2100000 | Edges Found: 1951
Lines: 2200000 | Edges Found: 2126
Lines: 2300000 | Edges Found: 2279
Lines: 2400000 | Edges Found: 2474
Complete! Extracted 2709 edges.


# CallOfDutyMobile_submissions subreddit

In [27]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/CallOfDutyMobile_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'CallOfDutyMobile')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("CallOfDutyMobile_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 286
Lines: 200000 | Edges Found: 574
Lines: 300000 | Edges Found: 785
Lines: 400000 | Edges Found: 970
Lines: 500000 | Edges Found: 1126
Lines: 600000 | Edges Found: 1381
Lines: 700000 | Edges Found: 1645
Lines: 800000 | Edges Found: 3442
Complete! Extracted 4876 edges.


# Christianity_submissions subreddit

In [28]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Christianity_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Christianity')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Christianity_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 3514
Lines: 300000 | Edges Found: 5858
Lines: 400000 | Edges Found: 7963
Lines: 500000 | Edges Found: 9263
Lines: 600000 | Edges Found: 10327
Lines: 700000 | Edges Found: 11842
Lines: 800000 | Edges Found: 13393
Lines: 900000 | Edges Found: 15020
Complete! Extracted 16307 edges.


# Conservative_submissions subreddit

In [29]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Conservative_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Conservative')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Conservative_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 342
Lines: 200000 | Edges Found: 1646
Lines: 300000 | Edges Found: 3198
Lines: 400000 | Edges Found: 5154
Lines: 500000 | Edges Found: 6816
Lines: 600000 | Edges Found: 8096
Lines: 700000 | Edges Found: 8829
Lines: 800000 | Edges Found: 9486
Lines: 900000 | Edges Found: 10192
Lines: 1000000 | Edges Found: 10895
Lines: 1100000 | Edges Found: 11558
Complete! Extracted 11936 edges.


# FIFA_submissions subreddit

In [30]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/FIFA_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'FIFA')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("FIFA_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 816
Lines: 200000 | Edges Found: 1746
Lines: 300000 | Edges Found: 2606
Lines: 400000 | Edges Found: 3311
Lines: 500000 | Edges Found: 3816
Lines: 600000 | Edges Found: 4194
Lines: 700000 | Edges Found: 4582
Lines: 800000 | Edges Found: 4916
Lines: 900000 | Edges Found: 5269
Lines: 1000000 | Edges Found: 5605
Lines: 1100000 | Edges Found: 5918
Lines: 1200000 | Edges Found: 6159
Lines: 1300000 | Edges Found: 6351
Lines: 1400000 | Edges Found: 6627
Lines: 1500000 | Edges Found: 6905
Lines: 1600000 | Edges Found: 7172
Lines: 1700000 | Edges Found: 7395
Lines: 1800000 | Edges Found: 7664
Lines: 1900000 | Edges Found: 7905
Lines: 2000000 | Edges Found: 8124
Complete! Extracted 8217 edges.


# nfl_submissions subreddit

In [31]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/nfl_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'nfl')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("nfl_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 4243
Lines: 300000 | Edges Found: 15646
Lines: 400000 | Edges Found: 30284
Lines: 500000 | Edges Found: 43787
Lines: 600000 | Edges Found: 59534
Lines: 700000 | Edges Found: 73203
Lines: 800000 | Edges Found: 91006
Lines: 900000 | Edges Found: 98561
Lines: 1000000 | Edges Found: 105845
Lines: 1100000 | Edges Found: 115559
Lines: 1200000 | Edges Found: 129658
Complete! Extracted 134213 edges.


# worldpolitics_submissions subreddit

In [32]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/worldpolitics_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'worldpolitics')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("worldpolitics_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 102
Lines: 200000 | Edges Found: 950
Lines: 300000 | Edges Found: 1511
Lines: 400000 | Edges Found: 2289
Lines: 500000 | Edges Found: 5831
Lines: 600000 | Edges Found: 8816
Lines: 700000 | Edges Found: 9750
Complete! Extracted 10038 edges.


# Android_submissions subreddit

In [33]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Android_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Android')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Android_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 2198
Lines: 400000 | Edges Found: 4949
Lines: 500000 | Edges Found: 6821
Lines: 600000 | Edges Found: 8548
Lines: 700000 | Edges Found: 9920
Lines: 800000 | Edges Found: 10896
Complete! Extracted 11063 edges.


# apple_submissions subreddit

In [34]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/apple_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'apple')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("apple_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 1683
Lines: 300000 | Edges Found: 3288
Lines: 400000 | Edges Found: 4505
Lines: 500000 | Edges Found: 5399
Lines: 600000 | Edges Found: 6079
Lines: 700000 | Edges Found: 6606
Lines: 800000 | Edges Found: 7038
Complete! Extracted 7098 edges.


# formula1_submissions subreddit

In [35]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/formula1_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'formula1')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("formula1_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1374
Lines: 200000 | Edges Found: 3697
Lines: 300000 | Edges Found: 5093
Lines: 400000 | Edges Found: 5878
Lines: 500000 | Edges Found: 6560
Lines: 600000 | Edges Found: 7310
Lines: 700000 | Edges Found: 8244
Complete! Extracted 9806 edges.


# Games_submissions subreddit

In [36]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Games_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Games')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Games_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 2250
Lines: 300000 | Edges Found: 4435
Lines: 400000 | Edges Found: 5881
Lines: 500000 | Edges Found: 7539
Lines: 600000 | Edges Found: 8834
Lines: 700000 | Edges Found: 10137
Lines: 800000 | Edges Found: 11207
Lines: 900000 | Edges Found: 12636
Complete! Extracted 13216 edges.


# HistoryMemes_submissions subreddit

In [37]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/HistoryMemes_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'HistoryMemes')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("HistoryMemes_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2096
Lines: 200000 | Edges Found: 2733
Lines: 300000 | Edges Found: 3495
Lines: 400000 | Edges Found: 3908
Lines: 500000 | Edges Found: 4307
Lines: 600000 | Edges Found: 4569
Lines: 700000 | Edges Found: 4824
Lines: 800000 | Edges Found: 5092
Complete! Extracted 5179 edges.


# islam_submissions subreddit

In [38]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/islam_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'islam')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("islam_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2971
Lines: 200000 | Edges Found: 4658
Lines: 300000 | Edges Found: 5792
Lines: 400000 | Edges Found: 7166
Lines: 500000 | Edges Found: 8257
Complete! Extracted 8622 edges.


# linux_submissions subreddit

In [39]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/linux_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'linux')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("linux_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1871
Lines: 200000 | Edges Found: 5606
Complete! Extracted 7766 edges.


# nintendo_submissions subreddit

In [40]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/nintendo_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'nintendo')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("nintendo_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 5665
Lines: 200000 | Edges Found: 26624
Complete! Extracted 27292 edges.


# nvidia_submissions subreddit

In [41]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/nvidia_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'nvidia')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("nvidia_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2073
Lines: 200000 | Edges Found: 3279
Lines: 300000 | Edges Found: 4659
Complete! Extracted 4917 edges.


# OutOfTheLoop_submissions subreddit

In [42]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/OutOfTheLoop_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'OutOfTheLoop')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("OutOfTheLoop_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 23128
Lines: 200000 | Edges Found: 39384
Lines: 300000 | Edges Found: 55635
Complete! Extracted 67338 edges.


# PoliticalHumor_submissions subreddit

In [43]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/PoliticalHumor_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'PoliticalHumor')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("PoliticalHumor_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 821
Lines: 200000 | Edges Found: 1912
Lines: 300000 | Edges Found: 2496
Lines: 400000 | Edges Found: 2875
Lines: 500000 | Edges Found: 3238
Lines: 600000 | Edges Found: 3660
Lines: 700000 | Edges Found: 4037
Lines: 800000 | Edges Found: 4282
Lines: 900000 | Edges Found: 4524
Complete! Extracted 4574 edges.


# science_submissions subreddit

In [44]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/science_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'science')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("science_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 179
Lines: 600000 | Edges Found: 620
Lines: 700000 | Edges Found: 891
Lines: 800000 | Edges Found: 1068
Lines: 900000 | Edges Found: 1218
Complete! Extracted 1240 edges.


# SubredditDrama_submissions subreddit

In [45]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/SubredditDrama_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'SubredditDrama')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("SubredditDrama_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 89759
Complete! Extracted 228606 edges.


# ukpolitics_submissions subreddit

In [46]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/ukpolitics_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ukpolitics')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("ukpolitics_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1013
Lines: 200000 | Edges Found: 1922
Lines: 300000 | Edges Found: 2705
Lines: 400000 | Edges Found: 3288
Lines: 500000 | Edges Found: 4298
Complete! Extracted 5214 edges.


# videogames_submissions subreddit

In [47]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/videogames_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'videogames')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("videogames_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 958
Lines: 200000 | Edges Found: 1968
Complete! Extracted 2435 edges.


# windows_submissions subreddit

In [48]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/windows_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'windows')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("windows_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1506
Complete! Extracted 2959 edges.


# AskReddit_submissions subreddit

In [49]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/AskReddit_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'AskReddit')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("AskReddit_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 0
Lines: 1600000 | Edges Found: 0
Lines: 1700000 | Edges Found: 0
Lines: 1800000 | Edges Found: 0
Lines: 1900000 | Edges Found: 0
Lines: 2000000 | Edges Found: 0
Lines: 2100000 | Edges Found: 0
Lines: 2200000 | Edges Found: 0
Lines: 2300000 | Edges Found: 0
Lines: 2400000 | Edges Found: 0
Lines: 2500000 | Edges Found: 0
Lines: 2600000 | Edges Found: 0
Lines: 2700000 | Edges Found: 0
Lines: 2800000 | Edges Found: 0
Lines: 2900000 | Edges Found: 0
Lines: 3000000 | Edges Found: 0
Lines: 3100000 | Edges Found: 0
Lines: 3200000 | 

# AutoNewspaper_submissions subreddit

In [50]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/AutoNewspaper_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'AutoNewspaper')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("AutoNewspaper_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 6
Lines: 200000 | Edges Found: 15
Lines: 300000 | Edges Found: 27
Lines: 400000 | Edges Found: 33
Lines: 500000 | Edges Found: 42
Lines: 600000 | Edges Found: 51
Lines: 700000 | Edges Found: 60
Lines: 800000 | Edges Found: 76
Lines: 900000 | Edges Found: 81
Lines: 1000000 | Edges Found: 81
Lines: 1100000 | Edges Found: 90
Lines: 1200000 | Edges Found: 99
Lines: 1300000 | Edges Found: 105
Lines: 1400000 | Edges Found: 109
Lines: 1500000 | Edges Found: 112
Lines: 1600000 | Edges Found: 117
Lines: 1700000 | Edges Found: 125
Lines: 1800000 | Edges Found: 134
Lines: 1900000 | Edges Found: 142
Lines: 2000000 | Edges Found: 155
Lines: 2100000 | Edges Found: 164
Lines: 2200000 | Edges Found: 174
Lines: 2300000 | Edges Found: 190
Lines: 2400000 | Edges Found: 195
Lines: 2500000 | Edges Found: 205
Lines: 2600000 | Edges Found: 212
Lines: 2700000 | Edges Found: 221
Lines: 2800000 | Edges Found: 223
Lines: 2900000 | Edges Found: 227
Lines: 3000000 | Edges Found: 230
Li

# aww_submissions subreddit

In [51]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/aww_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'aww')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("aww_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 516
Lines: 1300000 | Edges Found: 1357
Lines: 1400000 | Edges Found: 2229
Lines: 1500000 | Edges Found: 3042
Lines: 1600000 | Edges Found: 3847
Lines: 1700000 | Edges Found: 4686
Lines: 1800000 | Edges Found: 5415
Lines: 1900000 | Edges Found: 5992
Lines: 2000000 | Edges Found: 6581
Lines: 2100000 | Edges Found: 7064
Lines: 2200000 | Edges Found: 7575
Lines: 2300000 | Edges Found: 8068
Lines: 2400000 | Edges Found: 8500
Lines: 2500000 | Edges Found: 8936
Lines: 2600000 | Edges Found: 9395
Lines: 2700000 | Edges Found: 9802
Lines: 2800000 | Edges Found: 10199
Lines: 2900000 | Edges Found: 10547
Lines: 3000000 | Edges Fou

# funny_submissions subreddit

In [52]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/funny_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'funny')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("funny_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 0
Lines: 1600000 | Edges Found: 0
Lines: 1700000 | Edges Found: 0
Lines: 1800000 | Edges Found: 0
Lines: 1900000 | Edges Found: 0
Lines: 2000000 | Edges Found: 0
Lines: 2100000 | Edges Found: 0
Lines: 2200000 | Edges Found: 0
Lines: 2300000 | Edges Found: 0
Lines: 2400000 | Edges Found: 0
Lines: 2500000 | Edges Found: 0
Lines: 2600000 | Edges Found: 0
Lines: 2700000 | Edges Found: 0
Lines: 2800000 | Edges Found: 0
Lines: 2900000 | Edges Found: 0
Lines: 3000000 | Edges Found: 0
Lines: 3100000 | Edges Found: 0
Lines: 3200000 | 

# leagueoflegends_submissions subreddit

In [53]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/leagueoflegends_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'leagueoflegends')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("leagueoflegends_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 1473
Lines: 1100000 | Edges Found: 4339
Lines: 1200000 | Edges Found: 7059
Lines: 1300000 | Edges Found: 10230
Lines: 1400000 | Edges Found: 13239
Lines: 1500000 | Edges Found: 15867
Lines: 1600000 | Edges Found: 18780
Lines: 1700000 | Edges Found: 21610
Lines: 1800000 | Edges Found: 24501
Lines: 1900000 | Edges Found: 27458
Lines: 2000000 | Edges Found: 32192
Lines: 2100000 | Edges Found: 36425
Lines: 2200000 | Edges Found: 39335
Lines: 2300000 | Edges Found: 41933
Lines: 2400000 | Edges Found: 44652
Lines: 2500000 | Edges Found: 47133
Lines: 2600000 | Edges Found: 50018
Lines: 2700000 | Edges Found: 52296
Lines: 2800000 | Edges Found: 56299
Lines: 2900000 | Edges Found: 59404
Line

# memes_submissions subreddit

In [54]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/memes_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'memes')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("memes_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 272
Lines: 200000 | Edges Found: 626
Lines: 300000 | Edges Found: 1088
Lines: 400000 | Edges Found: 1476
Lines: 500000 | Edges Found: 1847
Lines: 600000 | Edges Found: 2279
Lines: 700000 | Edges Found: 2926
Lines: 800000 | Edges Found: 3329
Lines: 900000 | Edges Found: 3703
Lines: 1000000 | Edges Found: 4029
Lines: 1100000 | Edges Found: 4397
Lines: 1200000 | Edges Found: 5093
Lines: 1300000 | Edges Found: 5401
Lines: 1400000 | Edges Found: 5694
Lines: 1500000 | Edges Found: 5977
Lines: 1600000 | Edges Found: 6261
Lines: 1700000 | Edges Found: 6541
Lines: 1800000 | Edges Found: 6840
Lines: 1900000 | Edges Found: 7139
Lines: 2000000 | Edges Found: 7438
Lines: 2100000 | Edges Found: 7752
Lines: 2200000 | Edges Found: 7982
Lines: 2300000 | Edges Found: 8264
Lines: 2400000 | Edges Found: 8506
Lines: 2500000 | Edges Found: 8814
Lines: 2600000 | Edges Found: 9068
Lines: 2700000 | Edges Found: 9315
Lines: 2800000 | Edges Found: 9578
Lines: 2900000 | Edges Found: 9

# MonopolyGoTrading_submissions subreddit

In [55]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/MonopolyGoTrading_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'MonopolyGoTrading')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("MonopolyGoTrading_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 92
Lines: 200000 | Edges Found: 189
Lines: 300000 | Edges Found: 262
Lines: 400000 | Edges Found: 374
Lines: 500000 | Edges Found: 410
Lines: 600000 | Edges Found: 468
Lines: 700000 | Edges Found: 523
Lines: 800000 | Edges Found: 593
Lines: 900000 | Edges Found: 653
Lines: 1000000 | Edges Found: 724
Lines: 1100000 | Edges Found: 806
Lines: 1200000 | Edges Found: 862
Lines: 1300000 | Edges Found: 927
Lines: 1400000 | Edges Found: 1019
Complete! Extracted 1144 edges.


# teenagers_submissions subreddit

In [56]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/teenagers_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'teenagers')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("teenagers_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 1481
Lines: 300000 | Edges Found: 3701
Lines: 400000 | Edges Found: 5626
Lines: 500000 | Edges Found: 6856
Lines: 600000 | Edges Found: 7818
Lines: 700000 | Edges Found: 8710
Lines: 800000 | Edges Found: 9798
Lines: 900000 | Edges Found: 10963
Lines: 1000000 | Edges Found: 12139
Lines: 1100000 | Edges Found: 13494
Lines: 1200000 | Edges Found: 14818
Lines: 1300000 | Edges Found: 16330
Lines: 1400000 | Edges Found: 17829
Lines: 1500000 | Edges Found: 19062
Lines: 1600000 | Edges Found: 20432
Lines: 1700000 | Edges Found: 21609
Lines: 1800000 | Edges Found: 23096
Lines: 1900000 | Edges Found: 24599
Lines: 2000000 | Edges Found: 26129
Lines: 2100000 | Edges Found: 28076
Lines: 2200000 | Edges Found: 29372
Lines: 2300000 | Edges Found: 30748
Lines: 2400000 | Edges Found: 31779
Lines: 2500000 | Edges Found: 32862
Lines: 2600000 | Edges Found: 33819
Lines: 2700000 | Edges Found: 35087
Lines: 2800000 | Edges Found: 36402
Lines: 29000

# The_Donald_submissions subreddit

In [58]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/The_Donald_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'The_Donald')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("The_Donald_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 4040
Lines: 200000 | Edges Found: 8400
Lines: 300000 | Edges Found: 11944
Lines: 400000 | Edges Found: 18842
Lines: 500000 | Edges Found: 21950
Lines: 600000 | Edges Found: 25058
Lines: 700000 | Edges Found: 27365
Lines: 800000 | Edges Found: 29816
Lines: 900000 | Edges Found: 32234
Lines: 1000000 | Edges Found: 34416
Lines: 1100000 | Edges Found: 36497
Lines: 1200000 | Edges Found: 39029
Lines: 1300000 | Edges Found: 44183
Lines: 1400000 | Edges Found: 46937
Lines: 1500000 | Edges Found: 49152
Lines: 1600000 | Edges Found: 51664
Lines: 1700000 | Edges Found: 53949
Lines: 1800000 | Edges Found: 56948
Lines: 1900000 | Edges Found: 58797
Lines: 2000000 | Edges Found: 61327
Lines: 2100000 | Edges Found: 62865
Lines: 2200000 | Edges Found: 65803
Lines: 2300000 | Edges Found: 67363
Lines: 2400000 | Edges Found: 68878
Lines: 2500000 | Edges Found: 70070
Lines: 2600000 | Edges Found: 71035
Lines: 2700000 | Edges Found: 71818
Lines: 2800000 | Edges Found: 72616
Lin

# videos_submissions subreddit

In [59]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/videos_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'videos')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("videos_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 0
Lines: 1600000 | Edges Found: 491
Lines: 1700000 | Edges Found: 1385
Lines: 1800000 | Edges Found: 2324
Lines: 1900000 | Edges Found: 3226
Lines: 2000000 | Edges Found: 4091
Lines: 2100000 | Edges Found: 4980
Lines: 2200000 | Edges Found: 5824
Lines: 2300000 | Edges Found: 6733
Lines: 2400000 | Edges Found: 7542
Lines: 2500000 | Edges Found: 8214
Lines: 2600000 | Edges Found: 8816
Lines: 2700000 | Edges Found: 9419
Lines: 2800000 | Edges Found: 10010
Lines: 2900000 | Edges Found: 10485
Lines: 3000000 | Edges Found: 10943
Li

# buildapc_submissions subreddit

In [60]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/buildapc_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'buildapc')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("buildapc_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 3016
Lines: 500000 | Edges Found: 10972
Lines: 600000 | Edges Found: 19558
Lines: 700000 | Edges Found: 26883
Lines: 800000 | Edges Found: 33970
Lines: 900000 | Edges Found: 40175
Lines: 1000000 | Edges Found: 52957
Lines: 1100000 | Edges Found: 91680
Lines: 1200000 | Edges Found: 127831
Lines: 1300000 | Edges Found: 162398
Lines: 1400000 | Edges Found: 195072
Lines: 1500000 | Edges Found: 221458
Lines: 1600000 | Edges Found: 244116
Lines: 1700000 | Edges Found: 261180
Lines: 1800000 | Edges Found: 276614
Lines: 1900000 | Edges Found: 290826
Lines: 2000000 | Edges Found: 304798
Lines: 2100000 | Edges Found: 318901
Lines: 2200000 | Edges Found: 330371
Lines: 2300000 | Edges Found: 340231
Lines: 2400000 | Edges Found: 349309
Lines: 2500000 | Edges Found: 357622
Lines: 2600000 | Edges Found: 364086
Lines: 2700000 | Edges Found: 370225
Lines: 2800000 | Edges Found: 3760

# GlobalOffensiveTrade_submissions subreddit

In [61]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/GlobalOffensiveTrade_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'GlobalOffensiveTrade')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("GlobalOffensiveTrade_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2799
Lines: 200000 | Edges Found: 13853
Lines: 300000 | Edges Found: 42101
Lines: 400000 | Edges Found: 77370
Lines: 500000 | Edges Found: 117646
Lines: 600000 | Edges Found: 163414
Lines: 700000 | Edges Found: 214092
Lines: 800000 | Edges Found: 268500
Lines: 900000 | Edges Found: 324047
Lines: 1000000 | Edges Found: 374147
Lines: 1100000 | Edges Found: 423594
Lines: 1200000 | Edges Found: 475861
Lines: 1300000 | Edges Found: 528376
Lines: 1400000 | Edges Found: 581343
Lines: 1500000 | Edges Found: 634456
Lines: 1600000 | Edges Found: 688254
Lines: 1700000 | Edges Found: 741445
Lines: 1800000 | Edges Found: 793601
Lines: 1900000 | Edges Found: 846180
Lines: 2000000 | Edges Found: 898643
Lines: 2100000 | Edges Found: 950293
Lines: 2200000 | Edges Found: 1000920
Lines: 2300000 | Edges Found: 1049810
Lines: 2400000 | Edges Found: 1103322
Lines: 2500000 | Edges Found: 1152602
Lines: 2600000 | Edges Found: 1202410
Lines: 2700000 | Edges Found: 1251431
Lines: 28

# pics_submissions subreddit

In [62]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/pics_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'pics')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("pics_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 0
Lines: 1600000 | Edges Found: 0
Lines: 1700000 | Edges Found: 0
Lines: 1800000 | Edges Found: 0
Lines: 1900000 | Edges Found: 0
Lines: 2000000 | Edges Found: 0
Lines: 2100000 | Edges Found: 0
Lines: 2200000 | Edges Found: 0
Lines: 2300000 | Edges Found: 0
Lines: 2400000 | Edges Found: 0
Lines: 2500000 | Edges Found: 0
Lines: 2600000 | Edges Found: 0
Lines: 2700000 | Edges Found: 0
Lines: 2800000 | Edges Found: 0
Lines: 2900000 | Edges Found: 0
Lines: 3000000 | Edges Found: 0
Lines: 3100000 | Edges Found: 0
Lines: 3200000 | 

# relationship_advice_submissions subreddit

In [63]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/relationship_advice_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'relationship_advice')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("relationship_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 206
Lines: 200000 | Edges Found: 3586
Lines: 300000 | Edges Found: 5391
Lines: 400000 | Edges Found: 7005
Lines: 500000 | Edges Found: 8678
Lines: 600000 | Edges Found: 10438
Lines: 700000 | Edges Found: 12091
Lines: 800000 | Edges Found: 13580
Lines: 900000 | Edges Found: 14875
Lines: 1000000 | Edges Found: 16111
Lines: 1100000 | Edges Found: 17422
Lines: 1200000 | Edges Found: 18565
Lines: 1300000 | Edges Found: 19726
Lines: 1400000 | Edges Found: 20847
Lines: 1500000 | Edges Found: 21748
Lines: 1600000 | Edges Found: 22668
Lines: 1700000 | Edges Found: 23548
Lines: 1800000 | Edges Found: 24462
Lines: 1900000 | Edges Found: 25492
Lines: 2000000 | Edges Found: 26201
Lines: 2100000 | Edges Found: 26824
Lines: 2200000 | Edges Found: 27414
Lines: 2300000 | Edges Found: 28055
Lines: 2400000 | Edges Found: 28752
Lines: 2500000 | Edges Found: 29462
Lines: 2600000 | Edges Found: 30107
Lines: 2700000 | Edges Found: 30798
Lines: 2800000 | Edges Found: 31504
Lines: 

# Advice_submissions subreddit

In [64]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Advice_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Advice')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Advice_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3098
Lines: 200000 | Edges Found: 5407
Lines: 300000 | Edges Found: 7280
Lines: 400000 | Edges Found: 9065
Lines: 500000 | Edges Found: 10748
Lines: 600000 | Edges Found: 12430
Lines: 700000 | Edges Found: 13910
Lines: 800000 | Edges Found: 15345
Lines: 900000 | Edges Found: 16775
Lines: 1000000 | Edges Found: 18048
Lines: 1100000 | Edges Found: 19286
Lines: 1200000 | Edges Found: 20634
Lines: 1300000 | Edges Found: 21843
Lines: 1400000 | Edges Found: 22709
Lines: 1500000 | Edges Found: 23562
Lines: 1600000 | Edges Found: 24539
Lines: 1700000 | Edges Found: 25556
Lines: 1800000 | Edges Found: 26658
Lines: 1900000 | Edges Found: 27605
Lines: 2000000 | Edges Found: 28615
Lines: 2100000 | Edges Found: 29905
Lines: 2200000 | Edges Found: 31494
Lines: 2300000 | Edges Found: 33182
Lines: 2400000 | Edges Found: 34815
Lines: 2500000 | Edges Found: 36432
Lines: 2600000 | Edges Found: 37933
Complete! Extracted 38546 edges.


# cats_submissions subreddit

In [65]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/cats_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'cats')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("cats_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 688
Lines: 300000 | Edges Found: 2093
Lines: 400000 | Edges Found: 3106
Lines: 500000 | Edges Found: 3853
Lines: 600000 | Edges Found: 4349
Lines: 700000 | Edges Found: 4742
Lines: 800000 | Edges Found: 5076
Lines: 900000 | Edges Found: 5565
Lines: 1000000 | Edges Found: 5867
Lines: 1100000 | Edges Found: 6118
Lines: 1200000 | Edges Found: 6381
Lines: 1300000 | Edges Found: 6647
Lines: 1400000 | Edges Found: 6932
Lines: 1500000 | Edges Found: 7198
Lines: 1600000 | Edges Found: 7462
Lines: 1700000 | Edges Found: 7708
Lines: 1800000 | Edges Found: 8107
Lines: 1900000 | Edges Found: 8581
Lines: 2000000 | Edges Found: 8835
Lines: 2100000 | Edges Found: 9271
Lines: 2200000 | Edges Found: 9686
Lines: 2300000 | Edges Found: 10102
Lines: 2400000 | Edges Found: 10589
Lines: 2500000 | Edges Found: 11001
Lines: 2600000 | Edges Found: 11398
Lines: 2700000 | Edges Found: 11988
Lines: 2800000 | Edges Found: 12641
Lines: 2900000 | Edges Foun

# Market76_submissions subreddit

In [66]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Market76_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Market76')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Market76_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1594
Lines: 200000 | Edges Found: 3983
Lines: 300000 | Edges Found: 6104
Lines: 400000 | Edges Found: 7777
Lines: 500000 | Edges Found: 9459
Lines: 600000 | Edges Found: 11497
Lines: 700000 | Edges Found: 13406
Lines: 800000 | Edges Found: 16382
Lines: 900000 | Edges Found: 18694
Lines: 1000000 | Edges Found: 20829
Lines: 1100000 | Edges Found: 23956
Lines: 1200000 | Edges Found: 27086
Lines: 1300000 | Edges Found: 30918
Lines: 1400000 | Edges Found: 35025
Lines: 1500000 | Edges Found: 39575
Lines: 1600000 | Edges Found: 45361
Lines: 1700000 | Edges Found: 51737
Lines: 1800000 | Edges Found: 58298
Lines: 1900000 | Edges Found: 64789
Lines: 2000000 | Edges Found: 71454
Lines: 2100000 | Edges Found: 78746
Lines: 2200000 | Edges Found: 90501
Lines: 2300000 | Edges Found: 105608
Lines: 2400000 | Edges Found: 123937
Lines: 2500000 | Edges Found: 143039
Lines: 2600000 | Edges Found: 156295
Lines: 2700000 | Edges Found: 167352
Lines: 2800000 | Edges Found: 176931


# Market76_submissions subreddit

In [67]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Minecraft_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Minecraft')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Minecraft_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 915
Lines: 600000 | Edges Found: 2823
Lines: 700000 | Edges Found: 5227
Lines: 800000 | Edges Found: 6786
Lines: 900000 | Edges Found: 7602
Lines: 1000000 | Edges Found: 8137
Lines: 1100000 | Edges Found: 8687
Lines: 1200000 | Edges Found: 9198
Lines: 1300000 | Edges Found: 9722
Lines: 1400000 | Edges Found: 10290
Lines: 1500000 | Edges Found: 10738
Lines: 1600000 | Edges Found: 11182
Lines: 1700000 | Edges Found: 11664
Lines: 1800000 | Edges Found: 12073
Lines: 1900000 | Edges Found: 12519
Lines: 2000000 | Edges Found: 12951
Lines: 2100000 | Edges Found: 13417
Lines: 2200000 | Edges Found: 13848
Lines: 2300000 | Edges Found: 14294
Lines: 2400000 | Edges Found: 14705
Lines: 2500000 | Edges Found: 15168
Lines: 2600000 | Edges Found: 15759
Lines: 2700000 | Edges Found: 16681
Lines: 2800000 | Edges Found: 17339
Lines: 2900000 | Edges Foun

# Music_submissions subreddit

In [68]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Music_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Music')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Music_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 324
Lines: 1200000 | Edges Found: 2068
Lines: 1300000 | Edges Found: 3575
Lines: 1400000 | Edges Found: 5023
Lines: 1500000 | Edges Found: 6276
Lines: 1600000 | Edges Found: 7469
Lines: 1700000 | Edges Found: 8312
Lines: 1800000 | Edges Found: 9246
Lines: 1900000 | Edges Found: 10049
Lines: 2000000 | Edges Found: 10851
Lines: 2100000 | Edges Found: 12020
Lines: 2200000 | Edges Found: 13078
Lines: 2300000 | Edges Found: 14048
Lines: 2400000 | Edges Found: 14813
Lines: 2500000 | Edges Found: 15592
Lines: 2600000 | Edges Found: 16421
Lines: 2700000 | Edges Found: 17289
Lines: 2800000 | Edges Found: 17893
Lines: 2900000 | Edges Found: 18602
Lines: 3000000

# news_submissions subreddit

In [69]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/news_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'news')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("news_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 267
Lines: 800000 | Edges Found: 580
Lines: 900000 | Edges Found: 806
Lines: 1000000 | Edges Found: 1018
Lines: 1100000 | Edges Found: 1223
Lines: 1200000 | Edges Found: 1381
Lines: 1300000 | Edges Found: 1521
Lines: 1400000 | Edges Found: 1749
Lines: 1500000 | Edges Found: 1923
Lines: 1600000 | Edges Found: 2001
Lines: 1700000 | Edges Found: 2092
Lines: 1800000 | Edges Found: 2174
Lines: 1900000 | Edges Found: 2226
Lines: 2000000 | Edges Found: 2284
Lines: 2100000 | Edges Found: 2383
Lines: 2200000 | Edges Found: 2442
Lines: 2300000 | Edges Found: 2496
Lines: 2400000 | Edges Found: 2560
Lines: 2500000 | Edges Found: 2627
Lines: 2600000 | Edges Found: 2679
Lines: 2700000 | Edges Found: 2724
Lines: 2800000 | Edges Found: 2763
Lines: 2900000 | Edges Found: 2812
Lines: 3000000 

# PewdiepieSubmissions_submissions subreddit

In [70]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/PewdiepieSubmissions_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'PewdiepieSubmissions')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("PewdiepieSubmissions_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 144
Lines: 200000 | Edges Found: 350
Lines: 300000 | Edges Found: 628
Lines: 400000 | Edges Found: 947
Lines: 500000 | Edges Found: 1226
Lines: 600000 | Edges Found: 1458
Lines: 700000 | Edges Found: 1713
Lines: 800000 | Edges Found: 1963
Lines: 900000 | Edges Found: 2274
Lines: 1000000 | Edges Found: 2556
Lines: 1100000 | Edges Found: 3184
Lines: 1200000 | Edges Found: 3573
Lines: 1300000 | Edges Found: 3860
Lines: 1400000 | Edges Found: 4190
Lines: 1500000 | Edges Found: 4605
Lines: 1600000 | Edges Found: 4981
Lines: 1700000 | Edges Found: 5331
Lines: 1800000 | Edges Found: 5699
Lines: 1900000 | Edges Found: 5970
Lines: 2000000 | Edges Found: 6284
Lines: 2100000 | Edges Found: 6598
Lines: 2200000 | Edges Found: 6891
Lines: 2300000 | Edges Found: 7216
Lines: 2400000 | Edges Found: 7495
Lines: 2500000 | Edges Found: 7743
Lines: 2600000 | Edges Found: 7938
Lines: 2700000 | Edges Found: 8170
Lines: 2800000 | Edges Found: 8456
Lines: 2900000 | Edges Found: 887

# relationships_submissions subreddit

In [71]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/relationships_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'PewdiepieSubmissions')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("relationships_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 2590
Lines: 300000 | Edges Found: 6630
Lines: 400000 | Edges Found: 10284
Lines: 500000 | Edges Found: 12674
Lines: 600000 | Edges Found: 14825
Lines: 700000 | Edges Found: 16482
Lines: 800000 | Edges Found: 18035
Lines: 900000 | Edges Found: 19520
Lines: 1000000 | Edges Found: 20930
Lines: 1100000 | Edges Found: 21951
Lines: 1200000 | Edges Found: 22857
Lines: 1300000 | Edges Found: 23719
Lines: 1400000 | Edges Found: 24671
Lines: 1500000 | Edges Found: 25482
Lines: 1600000 | Edges Found: 26221
Lines: 1700000 | Edges Found: 26876
Lines: 1800000 | Edges Found: 27512
Lines: 1900000 | Edges Found: 28089
Lines: 2000000 | Edges Found: 28447
Lines: 2100000 | Edges Found: 28895
Lines: 2200000 | Edges Found: 29373
Lines: 2300000 | Edges Found: 29785
Lines: 2400000 | Edges Found: 30689
Lines: 2500000 | Edges Found: 31601
Lines: 2600000 | Edges Found: 32469
Lines: 2700000 | Edges Found: 33299
Complete! Extracted 33414 edges.


# 2007scape_submissions subreddit

In [72]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/2007scape_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', '2007scape')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("2007scape_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1243
Lines: 200000 | Edges Found: 2683
Lines: 300000 | Edges Found: 3891
Lines: 400000 | Edges Found: 5141
Lines: 500000 | Edges Found: 6501
Lines: 600000 | Edges Found: 7561
Lines: 700000 | Edges Found: 8493
Lines: 800000 | Edges Found: 9475
Lines: 900000 | Edges Found: 10444
Lines: 1000000 | Edges Found: 11285
Lines: 1100000 | Edges Found: 11993
Lines: 1200000 | Edges Found: 12872
Lines: 1300000 | Edges Found: 13966
Lines: 1400000 | Edges Found: 15025
Lines: 1500000 | Edges Found: 16092
Complete! Extracted 16751 edges.


# Accounting_submissions subreddit

In [73]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Accounting_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Accounting')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Accounting_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2053
Lines: 200000 | Edges Found: 3683
Lines: 300000 | Edges Found: 5104
Lines: 400000 | Edges Found: 6857
Complete! Extracted 6866 edges.


# Amazon__Deals__submissions subreddit

In [75]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Amazon__Deals__submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Amazon__Deals')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Amazon__Deals_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 45222
Complete! Extracted 54125 edges.


# animation_submissions subreddit

In [77]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/animation_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'animation')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("animation_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 970
Lines: 200000 | Edges Found: 1531
Complete! Extracted 2115 edges.


# anime_submissions subreddit

In [78]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/anime_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'anime')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("anime_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 346
Lines: 200000 | Edges Found: 6294
Lines: 300000 | Edges Found: 12927
Lines: 400000 | Edges Found: 16859
Lines: 500000 | Edges Found: 19994
Lines: 600000 | Edges Found: 23044
Lines: 700000 | Edges Found: 25672
Lines: 800000 | Edges Found: 28156
Lines: 900000 | Edges Found: 29489
Lines: 1000000 | Edges Found: 31245
Lines: 1100000 | Edges Found: 33117
Lines: 1200000 | Edges Found: 35282
Lines: 1300000 | Edges Found: 38054
Lines: 1400000 | Edges Found: 41539
Lines: 1500000 | Edges Found: 44788
Lines: 1600000 | Edges Found: 47099
Complete! Extracted 47853 edges.


# Anxiety_submissions subreddit

In [2]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Anxiety_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Anxiety')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Anxiety_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2482
Lines: 200000 | Edges Found: 4091
Lines: 300000 | Edges Found: 5329
Lines: 400000 | Edges Found: 6381
Lines: 500000 | Edges Found: 7208
Lines: 600000 | Edges Found: 8027
Lines: 700000 | Edges Found: 9295
Complete! Extracted 9943 edges.


# apexlegends_submissions subreddit

In [3]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/apexlegends_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'apexlegends')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("apexlegends_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 794
Lines: 200000 | Edges Found: 1438
Lines: 300000 | Edges Found: 1996
Lines: 400000 | Edges Found: 2554
Lines: 500000 | Edges Found: 3002
Lines: 600000 | Edges Found: 3475
Lines: 700000 | Edges Found: 3949
Lines: 800000 | Edges Found: 4408
Lines: 900000 | Edges Found: 4828
Lines: 1000000 | Edges Found: 5242
Lines: 1100000 | Edges Found: 5717
Lines: 1200000 | Edges Found: 6098
Lines: 1300000 | Edges Found: 6514
Lines: 1400000 | Edges Found: 6965
Lines: 1500000 | Edges Found: 7616
Lines: 1600000 | Edges Found: 8115
Lines: 1700000 | Edges Found: 8765
Complete! Extracted 9170 edges.


# Aquariums_submissions subreddit

In [4]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Aquariums_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Aquariums')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Aquariums_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1922
Lines: 200000 | Edges Found: 3728
Lines: 300000 | Edges Found: 4544
Lines: 400000 | Edges Found: 5233
Lines: 500000 | Edges Found: 5866
Lines: 600000 | Edges Found: 6367
Lines: 700000 | Edges Found: 6987
Lines: 800000 | Edges Found: 7836
Complete! Extracted 8728 edges.


# argentina_submissions subreddit

In [5]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/argentina_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'argentina')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("argentina_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3682
Lines: 200000 | Edges Found: 5804
Lines: 300000 | Edges Found: 7408
Lines: 400000 | Edges Found: 8769
Lines: 500000 | Edges Found: 10088
Complete! Extracted 10469 edges.


# Art_submissions subreddit

In [6]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Art_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Art')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Art_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 733
Lines: 300000 | Edges Found: 1512
Lines: 400000 | Edges Found: 1924
Lines: 500000 | Edges Found: 2346
Lines: 600000 | Edges Found: 2774
Lines: 700000 | Edges Found: 3180
Lines: 800000 | Edges Found: 3567
Lines: 900000 | Edges Found: 3867
Lines: 1000000 | Edges Found: 4274
Lines: 1100000 | Edges Found: 4695
Lines: 1200000 | Edges Found: 5121
Lines: 1300000 | Edges Found: 5571
Lines: 1400000 | Edges Found: 5950
Lines: 1500000 | Edges Found: 6289
Lines: 1600000 | Edges Found: 6616
Lines: 1700000 | Edges Found: 6869
Lines: 1800000 | Edges Found: 7255
Lines: 1900000 | Edges Found: 7651
Lines: 2000000 | Edges Found: 8066
Complete! Extracted 8243 edges.


# autism_submissions subreddit

In [7]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/autism_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'autism')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("autism_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1893
Lines: 200000 | Edges Found: 3299
Lines: 300000 | Edges Found: 4889
Complete! Extracted 6779 edges.


# Baking_submissions subreddit

In [8]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Baking_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Baking')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Baking_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1155
Lines: 200000 | Edges Found: 1644
Lines: 300000 | Edges Found: 2076
Lines: 400000 | Edges Found: 2796
Complete! Extracted 3132 edges.


# BeAmazed_submissions subreddit

In [9]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/BeAmazed_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'BeAmazed')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("BeAmazed_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 496
Lines: 200000 | Edges Found: 675
Complete! Extracted 731 edges.


# bipolar_submissions subreddit

In [10]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/bipolar_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'bipolar')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("bipolar_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2205
Lines: 200000 | Edges Found: 3478
Lines: 300000 | Edges Found: 4376
Lines: 400000 | Edges Found: 5167
Complete! Extracted 5254 edges.


# books_submissions subreddit

In [11]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/books_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'books')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("books_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 1748
Lines: 300000 | Edges Found: 3160
Lines: 400000 | Edges Found: 4480
Lines: 500000 | Edges Found: 6185
Lines: 600000 | Edges Found: 7724
Lines: 700000 | Edges Found: 8928
Lines: 800000 | Edges Found: 10301
Complete! Extracted 10567 edges.


# brasil_submissions subreddit

In [12]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/brasil_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'brasil')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("brasil_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3503
Lines: 200000 | Edges Found: 7861
Lines: 300000 | Edges Found: 11915
Lines: 400000 | Edges Found: 15324
Lines: 500000 | Edges Found: 18374
Lines: 600000 | Edges Found: 21472
Lines: 700000 | Edges Found: 25074
Complete! Extracted 27363 edges.


# buildapcforme_submissions subreddit

In [13]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/buildapcforme_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'buildapcforme')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("buildapcforme_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 175341
Lines: 200000 | Edges Found: 364337
Lines: 300000 | Edges Found: 522217
Complete! Extracted 614487 edges.


# careerguidance_submissions subreddit

In [14]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/careerguidance_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'careerguidance')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("careerguidance_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3721
Lines: 200000 | Edges Found: 5860
Lines: 300000 | Edges Found: 7747
Lines: 400000 | Edges Found: 9968
Lines: 500000 | Edges Found: 12368
Complete! Extracted 13902 edges.


# cars_submissions subreddit

In [15]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/cars_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'cars')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("cars_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 3321
Lines: 300000 | Edges Found: 5802
Lines: 400000 | Edges Found: 7866
Lines: 500000 | Edges Found: 10243
Lines: 600000 | Edges Found: 12136
Lines: 700000 | Edges Found: 14130
Lines: 800000 | Edges Found: 15786
Complete! Extracted 16181 edges.


# Catholicism_submissions subreddit

In [16]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Catholicism_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Catholicism')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Catholicism_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3205
Lines: 200000 | Edges Found: 5299
Lines: 300000 | Edges Found: 6606
Lines: 400000 | Edges Found: 7995
Complete! Extracted 8389 edges.


# Celebs_submissions subreddit

In [17]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Celebs_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Celebs')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Celebs_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 920
Lines: 200000 | Edges Found: 2055
Lines: 300000 | Edges Found: 2630
Lines: 400000 | Edges Found: 2975
Lines: 500000 | Edges Found: 3105
Lines: 600000 | Edges Found: 3215
Complete! Extracted 3255 edges.


# ChatGPT_submissions subreddit

In [18]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/ChatGPT_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ChatGPT')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("ChatGPT_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1322
Lines: 200000 | Edges Found: 2505
Lines: 300000 | Edges Found: 3225
Complete! Extracted 3281 edges.


# college_submissions subreddit

In [19]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/college_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ChatGPT')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("college_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1685
Lines: 200000 | Edges Found: 2904
Lines: 300000 | Edges Found: 3974
Lines: 400000 | Edges Found: 5203
Complete! Extracted 5983 edges.


# comics_submissions subreddit

In [20]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/comics_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ChatGPT')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("comics_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 95
Lines: 300000 | Edges Found: 698
Lines: 400000 | Edges Found: 1056
Lines: 500000 | Edges Found: 1409
Lines: 600000 | Edges Found: 1684
Lines: 700000 | Edges Found: 2015
Complete! Extracted 2541 edges.


# conspiracy_submissions subreddit

In [21]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/conspiracy_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ChatGPT')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("conspiracy_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 1723
Lines: 300000 | Edges Found: 7562
Lines: 400000 | Edges Found: 13282
Lines: 500000 | Edges Found: 18514
Lines: 600000 | Edges Found: 22805
Lines: 700000 | Edges Found: 27614
Lines: 800000 | Edges Found: 34194
Lines: 900000 | Edges Found: 39369
Lines: 1000000 | Edges Found: 43531
Lines: 1100000 | Edges Found: 46506
Lines: 1200000 | Edges Found: 50889
Lines: 1300000 | Edges Found: 53924
Lines: 1400000 | Edges Found: 55388
Lines: 1500000 | Edges Found: 56673
Lines: 1600000 | Edges Found: 57973
Lines: 1700000 | Edges Found: 59444
Complete! Extracted 59733 edges.


# Cricket_submissions subreddit

In [22]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Cricket_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Cricket')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Cricket_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3394
Lines: 200000 | Edges Found: 4836
Lines: 300000 | Edges Found: 6177
Lines: 400000 | Edges Found: 8409
Complete! Extracted 9714 edges.


# CryptoCurrencyClassic_submissions subreddit

In [23]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/CryptoCurrencyClassic_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'CryptoCurrencyClassic')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("CryptoCurrencyClassic_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 99345
Lines: 200000 | Edges Found: 198723
Lines: 300000 | Edges Found: 293875
Lines: 400000 | Edges Found: 387436
Lines: 500000 | Edges Found: 476181
Lines: 600000 | Edges Found: 570900
Lines: 700000 | Edges Found: 663952
Complete! Extracted 697977 edges.


# cyberpunkgame_submissions subreddit

In [24]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/cyberpunkgame_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'cyberpunkgame')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("cyberpunkgame_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 892
Lines: 200000 | Edges Found: 1616
Lines: 300000 | Edges Found: 2261
Lines: 400000 | Edges Found: 3062
Lines: 500000 | Edges Found: 4223
Complete! Extracted 4891 edges.


# Damnthatsinteresting_submissions subreddit

In [25]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Damnthatsinteresting_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Damnthatsinteresting')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Damnthatsinteresting_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1337
Lines: 200000 | Edges Found: 1584
Lines: 300000 | Edges Found: 1782
Lines: 400000 | Edges Found: 2082
Lines: 500000 | Edges Found: 2468
Complete! Extracted 2592 edges.


# dating_advice_submissions subreddit

In [26]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/dating_advice_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'dating_advice')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("dating_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1603
Lines: 200000 | Edges Found: 3124
Lines: 300000 | Edges Found: 4408
Lines: 400000 | Edges Found: 5398
Lines: 500000 | Edges Found: 6338
Lines: 600000 | Edges Found: 7035
Lines: 700000 | Edges Found: 7729
Lines: 800000 | Edges Found: 8488
Lines: 900000 | Edges Found: 9475
Lines: 1000000 | Edges Found: 10641
Lines: 1100000 | Edges Found: 11827
Complete! Extracted 12491 edges.


# deadbydaylight_submissions subreddit

In [27]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/deadbydaylight_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'deadbydaylight')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("deadbydaylight_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1463
Lines: 200000 | Edges Found: 2454
Lines: 300000 | Edges Found: 3262
Lines: 400000 | Edges Found: 4072
Lines: 500000 | Edges Found: 4707
Lines: 600000 | Edges Found: 5416
Lines: 700000 | Edges Found: 6143
Lines: 800000 | Edges Found: 6909
Lines: 900000 | Edges Found: 7664
Lines: 1000000 | Edges Found: 8441
Lines: 1100000 | Edges Found: 9392
Complete! Extracted 10221 edges.


# depression_submissions subreddit

In [28]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/depression_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'depression')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("depression_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 588
Lines: 200000 | Edges Found: 3967
Lines: 300000 | Edges Found: 5813
Lines: 400000 | Edges Found: 7197
Lines: 500000 | Edges Found: 8386
Lines: 600000 | Edges Found: 9387
Lines: 700000 | Edges Found: 10349
Lines: 800000 | Edges Found: 11334
Lines: 900000 | Edges Found: 12121
Lines: 1000000 | Edges Found: 12882
Lines: 1100000 | Edges Found: 13471
Lines: 1200000 | Edges Found: 14092
Lines: 1300000 | Edges Found: 14647
Lines: 1400000 | Edges Found: 15067
Lines: 1500000 | Edges Found: 15543
Lines: 1600000 | Edges Found: 16062
Lines: 1700000 | Edges Found: 16697
Complete! Extracted 17245 edges.


# DigitalArt_submissions subreddit

In [29]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/DigitalArt_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'DigitalArt')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("DigitalArt_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 422
Lines: 200000 | Edges Found: 764
Lines: 300000 | Edges Found: 1094
Complete! Extracted 1147 edges.


# DIY_submissions subreddit

In [30]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/DIY_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'DIY')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("DIY_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1846
Lines: 200000 | Edges Found: 4027
Lines: 300000 | Edges Found: 5183
Lines: 400000 | Edges Found: 6185
Lines: 500000 | Edges Found: 7339
Lines: 600000 | Edges Found: 9317
Complete! Extracted 10470 edges.


# dogecoin_submissions subreddit

In [31]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/dogecoin_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'dogecoin')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("dogecoin_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 5708
Lines: 200000 | Edges Found: 13320
Lines: 300000 | Edges Found: 14288
Lines: 400000 | Edges Found: 14907
Lines: 500000 | Edges Found: 15390
Lines: 600000 | Edges Found: 16008
Lines: 700000 | Edges Found: 16321
Lines: 800000 | Edges Found: 16721
Lines: 900000 | Edges Found: 17049
Lines: 1000000 | Edges Found: 17485
Lines: 1100000 | Edges Found: 18003
Lines: 1200000 | Edges Found: 18838
Lines: 1300000 | Edges Found: 22066
Complete! Extracted 22315 edges.


# drawing_submissions subreddit

In [32]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/drawing_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'drawing')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("drawing_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 890
Lines: 200000 | Edges Found: 1332
Lines: 300000 | Edges Found: 1894
Lines: 400000 | Edges Found: 2344
Lines: 500000 | Edges Found: 2686
Lines: 600000 | Edges Found: 2982
Lines: 700000 | Edges Found: 3208
Lines: 800000 | Edges Found: 3428
Lines: 900000 | Edges Found: 3632
Lines: 1000000 | Edges Found: 3859
Complete! Extracted 4116 edges.


# drums_submissions subreddit

In [33]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/drums_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'drums')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("drums_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1237
Lines: 200000 | Edges Found: 2230
Complete! Extracted 2883 edges.


# EarthPorn_submissions subreddit

In [34]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/EarthPorn_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'EarthPorn')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("EarthPorn_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 250
Lines: 200000 | Edges Found: 1134
Lines: 300000 | Edges Found: 1368
Lines: 400000 | Edges Found: 1569
Lines: 500000 | Edges Found: 1693
Lines: 600000 | Edges Found: 1755
Lines: 700000 | Edges Found: 1859
Lines: 800000 | Edges Found: 1956
Complete! Extracted 1960 edges.


# europe_submissions subreddit

In [35]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/europe_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'europe')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("europe_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2052
Lines: 200000 | Edges Found: 3772
Lines: 300000 | Edges Found: 5277
Lines: 400000 | Edges Found: 6172
Lines: 500000 | Edges Found: 6740
Lines: 600000 | Edges Found: 7184
Lines: 700000 | Edges Found: 7760
Lines: 800000 | Edges Found: 8239
Complete! Extracted 8302 edges.


# excel_submissions subreddit

In [36]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/excel_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'excel')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("excel_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1699
Lines: 200000 | Edges Found: 3042
Lines: 300000 | Edges Found: 4392
Complete! Extracted 5457 edges.


# facepalm_submissions subreddit

In [37]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/facepalm_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'facepalm')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("facepalm_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 604
Lines: 200000 | Edges Found: 2678
Lines: 300000 | Edges Found: 3949
Lines: 400000 | Edges Found: 5178
Lines: 500000 | Edges Found: 6105
Lines: 600000 | Edges Found: 6843
Lines: 700000 | Edges Found: 7499
Complete! Extracted 7519 edges.


# Fantasy_submissions subreddit

In [38]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Fantasy_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Fantasy')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Fantasy_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 4813
Lines: 200000 | Edges Found: 9857
Complete! Extracted 11186 edges.


# FashionReps_submissions subreddit

In [39]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/FashionReps_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'FashionReps')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("FashionReps_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2929
Lines: 200000 | Edges Found: 4395
Lines: 300000 | Edges Found: 5274
Lines: 400000 | Edges Found: 6007
Lines: 500000 | Edges Found: 6652
Lines: 600000 | Edges Found: 7366
Lines: 700000 | Edges Found: 8066
Lines: 800000 | Edges Found: 8645
Lines: 900000 | Edges Found: 9196
Lines: 1000000 | Edges Found: 9852
Lines: 1100000 | Edges Found: 10283
Lines: 1200000 | Edges Found: 10811
Lines: 1300000 | Edges Found: 11219
Lines: 1400000 | Edges Found: 11578
Lines: 1500000 | Edges Found: 12425
Lines: 1600000 | Edges Found: 13965
Lines: 1700000 | Edges Found: 15157
Complete! Extracted 16297 edges.


# food_submissions subreddit

In [40]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/food_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'food')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("food_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 558
Lines: 300000 | Edges Found: 1598
Lines: 400000 | Edges Found: 2465
Lines: 500000 | Edges Found: 2895
Lines: 600000 | Edges Found: 3140
Lines: 700000 | Edges Found: 3338
Lines: 800000 | Edges Found: 3473
Lines: 900000 | Edges Found: 3622
Lines: 1000000 | Edges Found: 3770
Lines: 1100000 | Edges Found: 3869
Lines: 1200000 | Edges Found: 3989
Lines: 1300000 | Edges Found: 4082
Lines: 1400000 | Edges Found: 4197
Lines: 1500000 | Edges Found: 4305
Lines: 1600000 | Edges Found: 4523
Complete! Extracted 4608 edges.


# FreeKarma4U_submissions subreddit

In [41]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/FreeKarma4U_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'FreeKarma4U')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("FreeKarma4U_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2595
Lines: 200000 | Edges Found: 4221
Lines: 300000 | Edges Found: 5503
Lines: 400000 | Edges Found: 7458
Lines: 500000 | Edges Found: 8957
Lines: 600000 | Edges Found: 10203
Lines: 700000 | Edges Found: 11396
Lines: 800000 | Edges Found: 12494
Lines: 900000 | Edges Found: 12929
Lines: 1000000 | Edges Found: 13585
Lines: 1100000 | Edges Found: 14069
Lines: 1200000 | Edges Found: 14782
Lines: 1300000 | Edges Found: 15047
Lines: 1400000 | Edges Found: 15067
Lines: 1500000 | Edges Found: 15083
Lines: 1600000 | Edges Found: 15104
Lines: 1700000 | Edges Found: 15134
Lines: 1800000 | Edges Found: 15153
Lines: 1900000 | Edges Found: 15178
Lines: 2000000 | Edges Found: 15389
Lines: 2100000 | Edges Found: 15479
Lines: 2200000 | Edges Found: 15532
Lines: 2300000 | Edges Found: 15587
Lines: 2400000 | Edges Found: 15646
Lines: 2500000 | Edges Found: 15699
Lines: 2600000 | Edges Found: 15756
Lines: 2700000 | Edges Found: 15799
Lines: 2800000 | Edges Found: 15860
Lines:

# gamedev_submissions subreddit

In [42]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/gamedev_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'gamedev')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("gamedev_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 6558
Lines: 200000 | Edges Found: 10232
Lines: 300000 | Edges Found: 13109
Complete! Extracted 14995 edges.


# gardening_submissions subreddit

In [43]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/gardening_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'gardening')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("gardening_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1288
Lines: 200000 | Edges Found: 2433
Lines: 300000 | Edges Found: 3240
Lines: 400000 | Edges Found: 4095
Lines: 500000 | Edges Found: 5617
Lines: 600000 | Edges Found: 6333
Lines: 700000 | Edges Found: 6773
Lines: 800000 | Edges Found: 7478
Lines: 900000 | Edges Found: 8224
Complete! Extracted 8874 edges.


# greece_submissions subreddit

In [44]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/greece_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'greece')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("greece_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2774
Lines: 200000 | Edges Found: 5124
Complete! Extracted 6587 edges.


# gtaonline_submissions subreddit

In [45]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/gtaonline_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'gtaonline')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("gtaonline_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2122
Lines: 200000 | Edges Found: 3325
Lines: 300000 | Edges Found: 4054
Lines: 400000 | Edges Found: 4560
Lines: 500000 | Edges Found: 5103
Lines: 600000 | Edges Found: 5520
Lines: 700000 | Edges Found: 5934
Lines: 800000 | Edges Found: 6377
Lines: 900000 | Edges Found: 6804
Lines: 1000000 | Edges Found: 7223
Lines: 1100000 | Edges Found: 7845
Lines: 1200000 | Edges Found: 8578
Complete! Extracted 8821 edges.


# Health_submissions subreddit

In [46]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Health_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Health')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Health_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 202
Lines: 400000 | Edges Found: 287
Lines: 500000 | Edges Found: 341
Lines: 600000 | Edges Found: 392
Lines: 700000 | Edges Found: 444
Lines: 800000 | Edges Found: 593
Lines: 900000 | Edges Found: 666
Complete! Extracted 672 edges.


# HolUp_submissions subreddit

In [47]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/HolUp_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'HolUp')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("HolUp_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 936
Lines: 200000 | Edges Found: 1791
Lines: 300000 | Edges Found: 2309
Lines: 400000 | Edges Found: 2682
Lines: 500000 | Edges Found: 2946
Lines: 600000 | Edges Found: 3230
Lines: 700000 | Edges Found: 3487
Complete! Extracted 3649 edges.


# houseplants_submissions subreddit

In [48]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/houseplants_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'houseplants')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("houseplants_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 543
Lines: 200000 | Edges Found: 980
Lines: 300000 | Edges Found: 1631
Lines: 400000 | Edges Found: 1967
Lines: 500000 | Edges Found: 2467
Lines: 600000 | Edges Found: 2875
Lines: 700000 | Edges Found: 3466
Lines: 800000 | Edges Found: 4268
Complete! Extracted 4283 edges.


# india_submissions subreddit

In [49]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/india_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'india')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("india_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 418
Lines: 200000 | Edges Found: 2636
Lines: 300000 | Edges Found: 4856
Lines: 400000 | Edges Found: 6645
Lines: 500000 | Edges Found: 7959
Lines: 600000 | Edges Found: 8933
Lines: 700000 | Edges Found: 9939
Lines: 800000 | Edges Found: 10866
Lines: 900000 | Edges Found: 11868
Lines: 1000000 | Edges Found: 12869
Lines: 1100000 | Edges Found: 14036
Lines: 1200000 | Edges Found: 15026
Lines: 1300000 | Edges Found: 16208
Lines: 1400000 | Edges Found: 17917
Lines: 1500000 | Edges Found: 18818
Lines: 1600000 | Edges Found: 20053
Lines: 1700000 | Edges Found: 20791
Complete! Extracted 21399 edges.


# ireland_submissions subreddit

In [50]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/ireland_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ireland')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("ireland_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2121
Lines: 200000 | Edges Found: 5438
Lines: 300000 | Edges Found: 8251
Lines: 400000 | Edges Found: 9093
Lines: 500000 | Edges Found: 10273
Lines: 600000 | Edges Found: 11216
Complete! Extracted 11365 edges.


# Jokes_submissions subreddit

In [51]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Jokes_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Jokes')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Jokes_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 318
Lines: 200000 | Edges Found: 912
Lines: 300000 | Edges Found: 1354
Lines: 400000 | Edges Found: 1659
Lines: 500000 | Edges Found: 1909
Lines: 600000 | Edges Found: 2147
Lines: 700000 | Edges Found: 2450
Lines: 800000 | Edges Found: 2679
Lines: 900000 | Edges Found: 2925
Lines: 1000000 | Edges Found: 3158
Lines: 1100000 | Edges Found: 3327
Lines: 1200000 | Edges Found: 4423
Lines: 1300000 | Edges Found: 4589
Lines: 1400000 | Edges Found: 4716
Lines: 1500000 | Edges Found: 4879
Lines: 1600000 | Edges Found: 5008
Complete! Extracted 5022 edges.


# kpop_submissions subreddit

In [52]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/kpop_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'kpop')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("kpop_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 11719
Lines: 200000 | Edges Found: 14437
Lines: 300000 | Edges Found: 16101
Lines: 400000 | Edges Found: 18930
Lines: 500000 | Edges Found: 20644
Complete! Extracted 21376 edges.


# ksi_submissions subreddit

In [53]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/ksi_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ksi')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("ksi_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 176
Lines: 200000 | Edges Found: 309
Lines: 300000 | Edges Found: 427
Lines: 400000 | Edges Found: 517
Lines: 500000 | Edges Found: 612
Lines: 600000 | Edges Found: 692
Lines: 700000 | Edges Found: 798
Lines: 800000 | Edges Found: 889
Lines: 900000 | Edges Found: 1023
Lines: 1000000 | Edges Found: 1187
Lines: 1100000 | Edges Found: 1297
Lines: 1200000 | Edges Found: 1405
Lines: 1300000 | Edges Found: 1542
Lines: 1400000 | Edges Found: 1640
Lines: 1500000 | Edges Found: 1749
Lines: 1600000 | Edges Found: 1863
Lines: 1700000 | Edges Found: 1939
Lines: 1800000 | Edges Found: 2037
Lines: 1900000 | Edges Found: 2281
Lines: 2000000 | Edges Found: 2388
Lines: 2100000 | Edges Found: 2488
Complete! Extracted 2670 edges.


# legaladvice_submissions subreddit

In [54]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/legaladvice_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'legaladvice')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("legaladvice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3999
Lines: 200000 | Edges Found: 8273
Lines: 300000 | Edges Found: 11599
Lines: 400000 | Edges Found: 14828
Lines: 500000 | Edges Found: 17780
Lines: 600000 | Edges Found: 20501
Lines: 700000 | Edges Found: 22938
Lines: 800000 | Edges Found: 25058
Lines: 900000 | Edges Found: 27101
Lines: 1000000 | Edges Found: 28863
Lines: 1100000 | Edges Found: 30542
Lines: 1200000 | Edges Found: 32322
Lines: 1300000 | Edges Found: 33820
Lines: 1400000 | Edges Found: 35432
Lines: 1500000 | Edges Found: 37130
Lines: 1600000 | Edges Found: 38760
Lines: 1700000 | Edges Found: 41188
Lines: 1800000 | Edges Found: 43659
Lines: 1900000 | Edges Found: 46039
Complete! Extracted 47430 edges.


# MadeMeSmile_submissions subreddit

In [55]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/MadeMeSmile_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'MadeMeSmile')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("MadeMeSmile_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2687
Lines: 200000 | Edges Found: 3560
Lines: 300000 | Edges Found: 4077
Lines: 400000 | Edges Found: 4403
Lines: 500000 | Edges Found: 4722
Lines: 600000 | Edges Found: 4953
Complete! Extracted 5021 edges.


# marvelstudios_submissions subreddit

In [56]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/marvelstudios_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'marvelstudios')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("marvelstudios_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2308
Lines: 200000 | Edges Found: 3701
Lines: 300000 | Edges Found: 4951
Lines: 400000 | Edges Found: 6098
Lines: 500000 | Edges Found: 7206
Lines: 600000 | Edges Found: 8135
Lines: 700000 | Edges Found: 9285
Complete! Extracted 10122 edges.


# me_irl_submissions subreddit

In [57]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/me_irl_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'me_irl')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("me_irl_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 60
Lines: 200000 | Edges Found: 120
Lines: 300000 | Edges Found: 194
Lines: 400000 | Edges Found: 247
Lines: 500000 | Edges Found: 316
Lines: 600000 | Edges Found: 375
Lines: 700000 | Edges Found: 445
Lines: 800000 | Edges Found: 506
Lines: 900000 | Edges Found: 566
Lines: 1000000 | Edges Found: 603
Lines: 1100000 | Edges Found: 645
Lines: 1200000 | Edges Found: 688
Lines: 1300000 | Edges Found: 729
Lines: 1400000 | Edges Found: 771
Lines: 1500000 | Edges Found: 820
Lines: 1600000 | Edges Found: 863
Lines: 1700000 | Edges Found: 997
Lines: 1800000 | Edges Found: 1046
Lines: 1900000 | Edges Found: 1091
Lines: 2000000 | Edges Found: 1144
Lines: 2100000 | Edges Found: 1189
Lines: 2200000 | Edges Found: 1241
Lines: 2300000 | Edges Found: 1335
Lines: 2400000 | Edges Found: 1413
Lines: 2500000 | Edges Found: 1484
Lines: 2600000 | Edges Found: 1560
Lines: 2700000 | Edges Found: 1640
Lines: 2800000 | Edges Found: 1732
Lines: 2900000 | Edges Found: 1817
Lines: 30000

# medical_advice_submissions subreddit

In [58]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/medical_advice_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'medical_advice')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("medical_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 963
Lines: 200000 | Edges Found: 1788
Lines: 300000 | Edges Found: 2992
Complete! Extracted 3790 edges.


# medical_submissions subreddit

In [59]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/medical_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'medical_submissions')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("medical_submissions_advice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1378
Lines: 200000 | Edges Found: 2367
Lines: 300000 | Edges Found: 3128
Lines: 400000 | Edges Found: 4517
Complete! Extracted 4647 edges.


# meme_submissions subreddit

In [60]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/meme_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'meme')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("meme_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 360
Lines: 200000 | Edges Found: 669
Lines: 300000 | Edges Found: 1107
Lines: 400000 | Edges Found: 1506
Lines: 500000 | Edges Found: 1932
Lines: 600000 | Edges Found: 2320
Lines: 700000 | Edges Found: 2742
Lines: 800000 | Edges Found: 3122
Lines: 900000 | Edges Found: 3453
Lines: 1000000 | Edges Found: 3804
Lines: 1100000 | Edges Found: 4129
Lines: 1200000 | Edges Found: 4436
Lines: 1300000 | Edges Found: 4675
Lines: 1400000 | Edges Found: 4979
Lines: 1500000 | Edges Found: 5276
Lines: 1600000 | Edges Found: 5441
Complete! Extracted 5549 edges.


# Memes_Of_The_Dank_submissions subreddit

In [61]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Memes_Of_The_Dank_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Memes_Of_The_Dank')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Memes_Of_The_Dank_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 542
Lines: 200000 | Edges Found: 1055
Lines: 300000 | Edges Found: 1830
Lines: 400000 | Edges Found: 2533
Lines: 500000 | Edges Found: 2962
Lines: 600000 | Edges Found: 3222
Lines: 700000 | Edges Found: 3347
Complete! Extracted 3361 edges.


# mexico_submissions subreddit

In [62]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/mexico_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'mexico')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("mexico_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1905
Lines: 200000 | Edges Found: 3494
Lines: 300000 | Edges Found: 4893
Lines: 400000 | Edges Found: 5970
Complete! Extracted 6234 edges.


# midjourney_submissions subreddit

In [63]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/midjourney_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'midjourney')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("midjourney_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 431
Complete! Extracted 855 edges.


# mildlyinfuriating_submissions subreddit

In [64]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/mildlyinfuriating_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'mildlyinfuriating')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("mildlyinfuriating_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1744
Lines: 200000 | Edges Found: 3482
Lines: 300000 | Edges Found: 4851
Lines: 400000 | Edges Found: 6082
Lines: 500000 | Edges Found: 7112
Lines: 600000 | Edges Found: 7918
Lines: 700000 | Edges Found: 8618
Lines: 800000 | Edges Found: 9453
Lines: 900000 | Edges Found: 10185
Lines: 1000000 | Edges Found: 11103
Lines: 1100000 | Edges Found: 12055
Lines: 1200000 | Edges Found: 13033
Lines: 1300000 | Edges Found: 13795
Lines: 1400000 | Edges Found: 14450
Lines: 1500000 | Edges Found: 14901
Complete! Extracted 14978 edges.


# mildlyinteresting_submissions subreddit

In [65]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/mildlyinteresting_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'mildlyinteresting')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("mildlyinteresting_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 982
Lines: 300000 | Edges Found: 1819
Lines: 400000 | Edges Found: 2467
Lines: 500000 | Edges Found: 2915
Lines: 600000 | Edges Found: 3327
Lines: 700000 | Edges Found: 3754
Lines: 800000 | Edges Found: 4177
Lines: 900000 | Edges Found: 4575
Lines: 1000000 | Edges Found: 4918
Lines: 1100000 | Edges Found: 5300
Lines: 1200000 | Edges Found: 5592
Lines: 1300000 | Edges Found: 5887
Lines: 1400000 | Edges Found: 6150
Lines: 1500000 | Edges Found: 6387
Lines: 1600000 | Edges Found: 6615
Lines: 1700000 | Edges Found: 6862
Lines: 1800000 | Edges Found: 7057
Lines: 1900000 | Edges Found: 7290
Lines: 2000000 | Edges Found: 7546
Lines: 2100000 | Edges Found: 7771
Complete! Extracted 7878 edges.


# MortalKombat_submissions subreddit

In [66]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/MortalKombat_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'MortalKombat')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("MortalKombat_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1167
Lines: 200000 | Edges Found: 1921
Lines: 300000 | Edges Found: 2479
Lines: 400000 | Edges Found: 3180
Complete! Extracted 3714 edges.


# movies_submissions subreddit

In [67]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/movies_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'movies')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("movies_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 634
Lines: 600000 | Edges Found: 2183
Lines: 700000 | Edges Found: 3658
Lines: 800000 | Edges Found: 5085
Lines: 900000 | Edges Found: 6410
Lines: 1000000 | Edges Found: 7718
Lines: 1100000 | Edges Found: 8735
Lines: 1200000 | Edges Found: 9580
Lines: 1300000 | Edges Found: 10330
Lines: 1400000 | Edges Found: 11172
Lines: 1500000 | Edges Found: 11980
Lines: 1600000 | Edges Found: 12993
Lines: 1700000 | Edges Found: 14031
Lines: 1800000 | Edges Found: 15037
Lines: 1900000 | Edges Found: 15933
Lines: 2000000 | Edges Found: 17102
Lines: 2100000 | Edges Found: 18134
Lines: 2200000 | Edges Found: 19268
Lines: 2300000 | Edges Found: 20625
Lines: 2400000 | Edges Found: 22325
Lines: 2500000 | Edges Found: 23738
Lines: 2600000 | Edges Found: 25575
Lines: 2700000 | Edges Found: 27153
Complete! Extracted 27255 edges.


# newsfeedmedia_submissions subreddit

In [68]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/newsfeedmedia_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'newsfeedmedia')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("newsfeedmedia_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2
Lines: 200000 | Edges Found: 3
Lines: 300000 | Edges Found: 6
Lines: 400000 | Edges Found: 7
Complete! Extracted 7 edges.


# newzealand_submissions subreddit

In [69]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/newzealand_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'newzealand')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("newzealand_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2704
Lines: 200000 | Edges Found: 4092
Lines: 300000 | Edges Found: 5444
Complete! Extracted 6945 edges.


# nostalgia_submissions subreddit

In [70]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/nostalgia_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'nostalgia')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("nostalgia_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 630
Lines: 200000 | Edges Found: 930
Lines: 300000 | Edges Found: 1215
Lines: 400000 | Edges Found: 1473
Complete! Extracted 1568 edges.


# NoStupidQuestions_submissions subreddit

In [71]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/NoStupidQuestions_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'NoStupidQuestions')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("NoStupidQuestions_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3762
Lines: 200000 | Edges Found: 7089
Lines: 300000 | Edges Found: 10256
Lines: 400000 | Edges Found: 13007
Lines: 500000 | Edges Found: 15677
Lines: 600000 | Edges Found: 18533
Lines: 700000 | Edges Found: 21080
Lines: 800000 | Edges Found: 23766
Lines: 900000 | Edges Found: 26372
Lines: 1000000 | Edges Found: 28702
Lines: 1100000 | Edges Found: 30978
Lines: 1200000 | Edges Found: 33054
Lines: 1300000 | Edges Found: 35114
Lines: 1400000 | Edges Found: 37003
Lines: 1500000 | Edges Found: 39125
Lines: 1600000 | Edges Found: 41588
Lines: 1700000 | Edges Found: 43562
Lines: 1800000 | Edges Found: 45370
Lines: 1900000 | Edges Found: 47334
Lines: 2000000 | Edges Found: 49238
Lines: 2100000 | Edges Found: 50940
Lines: 2200000 | Edges Found: 52426
Lines: 2300000 | Edges Found: 53734
Lines: 2400000 | Edges Found: 55114
Lines: 2500000 | Edges Found: 56574
Lines: 2600000 | Edges Found: 57906
Lines: 2700000 | Edges Found: 59237
Lines: 2800000 | Edges Found: 60474
Lin

# oddlysatisfying_submissions subreddit

In [72]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/oddlysatisfying_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'oddlysatisfying')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("oddlysatisfying_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 5966
Lines: 200000 | Edges Found: 7266
Lines: 300000 | Edges Found: 7884
Lines: 400000 | Edges Found: 8291
Lines: 500000 | Edges Found: 8694
Lines: 600000 | Edges Found: 9109
Complete! Extracted 9324 edges.


# pakistan_submissions subreddit

In [73]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/pakistan_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'pakistan')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("pakistan_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1594
Lines: 200000 | Edges Found: 2930
Lines: 300000 | Edges Found: 4126
Complete! Extracted 4479 edges.


 # Parenting_submissions subreddit

In [74]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Parenting_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Parenting')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Parenting_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2153
Lines: 200000 | Edges Found: 4103
Lines: 300000 | Edges Found: 5622
Lines: 400000 | Edges Found: 7060
Lines: 500000 | Edges Found: 8763
Complete! Extracted 8771 edges.


 # PcBuild_submissions subreddit

In [75]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/PcBuild_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'PcBuild')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("PcBuild_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3780
Lines: 200000 | Edges Found: 6903
Lines: 300000 | Edges Found: 10098
Lines: 400000 | Edges Found: 13433
Complete! Extracted 13905 edges.


 # pcgaming_submissions subreddit

In [76]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/pcgaming_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'pcgaming')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("pcgaming_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2549
Lines: 200000 | Edges Found: 4369
Lines: 300000 | Edges Found: 6096
Lines: 400000 | Edges Found: 7141
Lines: 500000 | Edges Found: 8229
Complete! Extracted 8269 edges.


 # personalfinance_submissions subreddit

In [77]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/personalfinance_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'personalfinance')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("personalfinance_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2782
Lines: 200000 | Edges Found: 6998
Lines: 300000 | Edges Found: 10147
Lines: 400000 | Edges Found: 13029
Lines: 500000 | Edges Found: 15855
Lines: 600000 | Edges Found: 18323
Lines: 700000 | Edges Found: 20824
Lines: 800000 | Edges Found: 23096
Lines: 900000 | Edges Found: 24973
Lines: 1000000 | Edges Found: 26466
Lines: 1100000 | Edges Found: 28021
Lines: 1200000 | Edges Found: 29557
Lines: 1300000 | Edges Found: 30977
Lines: 1400000 | Edges Found: 32421
Lines: 1500000 | Edges Found: 34133
Lines: 1600000 | Edges Found: 35910
Complete! Extracted 35974 edges.


 # Philippines_submissions subreddit

In [78]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Philippines_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Philippines')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Philippines_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2993
Lines: 200000 | Edges Found: 5924
Lines: 300000 | Edges Found: 8049
Lines: 400000 | Edges Found: 9434
Lines: 500000 | Edges Found: 10371
Lines: 600000 | Edges Found: 11673
Lines: 700000 | Edges Found: 12886
Lines: 800000 | Edges Found: 14332
Lines: 900000 | Edges Found: 15335
Complete! Extracted 15684 edges.


 # playstation_submissions subreddit

In [79]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/playstation_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'playstation')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("playstation_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 8941
Lines: 200000 | Edges Found: 9483
Lines: 300000 | Edges Found: 9969
Lines: 400000 | Edges Found: 10416
Lines: 500000 | Edges Found: 10949
Complete! Extracted 11209 edges.


 # pokemon_submissions subreddit

In [80]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/pokemon_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'pokemon')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("pokemon_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 956
Lines: 400000 | Edges Found: 3955
Lines: 500000 | Edges Found: 6540
Lines: 600000 | Edges Found: 9571
Lines: 700000 | Edges Found: 12814
Lines: 800000 | Edges Found: 14361
Lines: 900000 | Edges Found: 15867
Lines: 1000000 | Edges Found: 17400
Lines: 1100000 | Edges Found: 18954
Lines: 1200000 | Edges Found: 20614
Lines: 1300000 | Edges Found: 22590
Lines: 1400000 | Edges Found: 24393
Lines: 1500000 | Edges Found: 27095
Complete! Extracted 28565 edges.


 # POLITIC_submissions subreddit

In [81]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/POLITIC_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'POLITIC')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("POLITIC_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 839
Lines: 1300000 | Edges Found: 1929
Lines: 1400000 | Edges Found: 3120
Lines: 1500000 | Edges Found: 4768
Lines: 1600000 | Edges Found: 5935
Lines: 1700000 | Edges Found: 6452
Lines: 1800000 | Edges Found: 6892
Complete! Extracted 6917 edges.


 # PrequelMemes_submissions subreddit

In [82]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/PrequelMemes_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'PrequelMemes')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("PrequelMemes_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3832
Lines: 200000 | Edges Found: 7224
Lines: 300000 | Edges Found: 10158
Lines: 400000 | Edges Found: 12806
Lines: 500000 | Edges Found: 13911
Lines: 600000 | Edges Found: 14885
Lines: 700000 | Edges Found: 15680
Lines: 800000 | Edges Found: 16734
Lines: 900000 | Edges Found: 17402
Lines: 1000000 | Edges Found: 18078
Lines: 1100000 | Edges Found: 18636
Complete! Extracted 19116 edges.


 # PS4_submissions subreddit

In [83]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/PS4_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'PS4')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("PS4_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1527
Lines: 200000 | Edges Found: 3446
Lines: 300000 | Edges Found: 4970
Lines: 400000 | Edges Found: 6379
Lines: 500000 | Edges Found: 7440
Lines: 600000 | Edges Found: 8354
Lines: 700000 | Edges Found: 8987
Lines: 800000 | Edges Found: 9677
Lines: 900000 | Edges Found: 10027
Complete! Extracted 10198 edges.


 # Rabbits_submissions subreddit

In [84]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Rabbits_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Rabbits')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Rabbits_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 763
Lines: 200000 | Edges Found: 1148
Lines: 300000 | Edges Found: 1585
Lines: 400000 | Edges Found: 2088
Complete! Extracted 2294 edges.


 # reddeadredemption_submissions subreddit

In [85]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/reddeadredemption_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'reddeadredemption')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("reddeadredemption_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 699
Lines: 200000 | Edges Found: 1291
Lines: 300000 | Edges Found: 1866
Lines: 400000 | Edges Found: 2368
Lines: 500000 | Edges Found: 2841
Lines: 600000 | Edges Found: 3213
Lines: 700000 | Edges Found: 3793
Complete! Extracted 4295 edges.


 # reddit.com_submissions subreddit

In [87]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/reddit.com_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'reddit')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("reddit.com_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 0
Lines: 1600000 | Edges Found: 0
Lines: 1700000 | Edges Found: 0
Lines: 1800000 | Edges Found: 0
Lines: 1900000 | Edges Found: 0
Lines: 2000000 | Edges Found: 0
Lines: 2100000 | Edges Found: 0
Lines: 2200000 | Edges Found: 0
Lines: 2300000 | Edges Found: 0
Lines: 2400000 | Edges Found: 0
Lines: 2500000 | Edges Found: 0
Lines: 2600000 | Edges Found: 0
Lines: 2700000 | Edges Found: 0
Lines: 2800000 | Edges Found: 0
Lines: 2900000 | Edges Found: 0
Lines: 3000000 | Edges Found: 0
Lines: 3100000 | Edges Found: 0
Lines: 3200000 | 

 # Showerthoughts_submissions subreddit

In [88]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Showerthoughts_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Showerthoughts')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Showerthoughts_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1305
Lines: 200000 | Edges Found: 2762
Lines: 300000 | Edges Found: 4562
Lines: 400000 | Edges Found: 6564
Lines: 500000 | Edges Found: 8984
Lines: 600000 | Edges Found: 11315
Lines: 700000 | Edges Found: 13114
Lines: 800000 | Edges Found: 15056
Lines: 900000 | Edges Found: 17232
Lines: 1000000 | Edges Found: 19138
Lines: 1100000 | Edges Found: 20905
Lines: 1200000 | Edges Found: 22679
Lines: 1300000 | Edges Found: 24662
Lines: 1400000 | Edges Found: 26465
Lines: 1500000 | Edges Found: 28154
Lines: 1600000 | Edges Found: 29855
Lines: 1700000 | Edges Found: 31213
Lines: 1800000 | Edges Found: 32588
Lines: 1900000 | Edges Found: 33782
Lines: 2000000 | Edges Found: 34870
Lines: 2100000 | Edges Found: 35942
Lines: 2200000 | Edges Found: 37006
Lines: 2300000 | Edges Found: 38104
Lines: 2400000 | Edges Found: 39053
Lines: 2500000 | Edges Found: 40152
Lines: 2600000 | Edges Found: 41089
Lines: 2700000 | Edges Found: 41924
Lines: 2800000 | Edges Found: 42882
Lines:

 # skyrimmods_submissions subreddit

In [89]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/skyrimmods_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'skyrimmods')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("skyrimmods_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3310
Lines: 200000 | Edges Found: 6936
Lines: 300000 | Edges Found: 9530
Lines: 400000 | Edges Found: 11881
Complete! Extracted 12903 edges.


 # Sneakers_submissions subreddit

In [90]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Sneakers_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Sneakers')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Sneakers_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 662
Lines: 200000 | Edges Found: 1463
Lines: 300000 | Edges Found: 2050
Lines: 400000 | Edges Found: 2530
Lines: 500000 | Edges Found: 2921
Lines: 600000 | Edges Found: 3236
Lines: 700000 | Edges Found: 3525
Lines: 800000 | Edges Found: 3740
Lines: 900000 | Edges Found: 3945
Lines: 1000000 | Edges Found: 4170
Lines: 1100000 | Edges Found: 4483
Complete! Extracted 4764 edges.


 # softwaregore_submissions subreddit

In [91]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/softwaregore_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'softwaregore')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("softwaregore_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1300
Lines: 200000 | Edges Found: 1864
Lines: 300000 | Edges Found: 2333
Lines: 400000 | Edges Found: 2691
Lines: 500000 | Edges Found: 3008
Lines: 600000 | Edges Found: 3338
Lines: 700000 | Edges Found: 3667
Lines: 800000 | Edges Found: 3948
Complete! Extracted 4196 edges.


 # space_submissions subreddit

In [ ]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/space_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'space')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("space_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


 # Spiderman_submissions subreddit

In [93]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Spiderman_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Spiderman')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Spiderman_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1314
Lines: 200000 | Edges Found: 1982
Complete! Extracted 3048 edges.


 # sports_submissions subreddit

In [94]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/sports_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'sports')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("sports_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 150
Lines: 300000 | Edges Found: 1192
Lines: 400000 | Edges Found: 1969
Lines: 500000 | Edges Found: 2468
Lines: 600000 | Edges Found: 3043
Lines: 700000 | Edges Found: 3814
Lines: 800000 | Edges Found: 4043
Complete! Extracted 4069 edges.


 # spotify_submissions subreddit

In [95]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/spotify_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'spotify')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("spotify_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 8152
Lines: 200000 | Edges Found: 11168
Lines: 300000 | Edges Found: 11793
Lines: 400000 | Edges Found: 12413
Complete! Extracted 12942 edges.


 # SpotifyPlaylists_submissions subreddit

In [96]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/SpotifyPlaylists_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'SpotifyPlaylists')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("SpotifyPlaylists_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 391
Lines: 200000 | Edges Found: 600
Lines: 300000 | Edges Found: 790
Complete! Extracted 931 edges.


 # StarWars_submissions subreddit

In [97]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/StarWars_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'StarWars')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("StarWars_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1297
Lines: 200000 | Edges Found: 3170
Lines: 300000 | Edges Found: 4720
Lines: 400000 | Edges Found: 5862
Lines: 500000 | Edges Found: 6702
Lines: 600000 | Edges Found: 7358
Lines: 700000 | Edges Found: 8047
Lines: 800000 | Edges Found: 8797
Lines: 900000 | Edges Found: 9682
Complete! Extracted 10240 edges.


 # StreetFighter_submissions subreddit

In [98]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/StreetFighter_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'StreetFighter')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("StreetFighter_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 39468
Lines: 200000 | Edges Found: 69066
Complete! Extracted 70173 edges.


 # sweden_submissions subreddit

In [99]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/sweden_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'sweden')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("sweden_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3286
Lines: 200000 | Edges Found: 7273
Lines: 300000 | Edges Found: 10216
Lines: 400000 | Edges Found: 13892
Complete! Extracted 16864 edges.


 # TalkativePeople_submissions subreddit

In [1]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/TalkativePeople_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'TalkativePeople')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("TalkativePeople_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 15245
Lines: 200000 | Edges Found: 18390
Lines: 300000 | Edges Found: 21961
Lines: 400000 | Edges Found: 27775
Lines: 500000 | Edges Found: 37923
Lines: 600000 | Edges Found: 38478
Lines: 700000 | Edges Found: 38766
Lines: 800000 | Edges Found: 39344
Lines: 900000 | Edges Found: 39817
Lines: 1000000 | Edges Found: 40288
Lines: 1100000 | Edges Found: 41064
Lines: 1200000 | Edges Found: 41663
Lines: 1300000 | Edges Found: 42106
Lines: 1400000 | Edges Found: 45385
Lines: 1500000 | Edges Found: 45841
Lines: 1600000 | Edges Found: 46672
Lines: 1700000 | Edges Found: 47042
Lines: 1800000 | Edges Found: 47400
Lines: 1900000 | Edges Found: 48049
Lines: 2000000 | Edges Found: 49112
Lines: 2100000 | Edges Found: 49922
Lines: 2200000 | Edges Found: 50420
Lines: 2300000 | Edges Found: 50961
Lines: 2400000 | Edges Found: 51384
Lines: 2500000 | Edges Found: 52084
Lines: 2600000 | Edges Found: 52851
Lines: 2700000 | Edges Found: 53249
Lines: 2800000 | Edges Found: 53744
L

 # techsupport_submissions subreddit

In [2]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/techsupport_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'techsupport')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("techsupport_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 1191
Lines: 300000 | Edges Found: 6468
Lines: 400000 | Edges Found: 11332
Lines: 500000 | Edges Found: 15198
Lines: 600000 | Edges Found: 19299
Lines: 700000 | Edges Found: 23216
Lines: 800000 | Edges Found: 26599
Lines: 900000 | Edges Found: 29830
Lines: 1000000 | Edges Found: 32926
Lines: 1100000 | Edges Found: 35683
Lines: 1200000 | Edges Found: 38061
Lines: 1300000 | Edges Found: 40291
Lines: 1400000 | Edges Found: 42806
Lines: 1500000 | Edges Found: 45110
Lines: 1600000 | Edges Found: 47217
Lines: 1700000 | Edges Found: 49431
Lines: 1800000 | Edges Found: 51806
Lines: 1900000 | Edges Found: 54171
Lines: 2000000 | Edges Found: 56664
Lines: 2100000 | Edges Found: 59368
Complete! Extracted 61374 edges.


 # TheNewsFeed_submissions subreddit

In [3]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/TheNewsFeed_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'TheNewsFeed')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("TheNewsFeed_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 7
Lines: 200000 | Edges Found: 12
Lines: 300000 | Edges Found: 15
Lines: 400000 | Edges Found: 19
Lines: 500000 | Edges Found: 23
Lines: 600000 | Edges Found: 26
Lines: 700000 | Edges Found: 32
Lines: 800000 | Edges Found: 47
Lines: 900000 | Edges Found: 58
Complete! Extracted 68 edges.


 # todayilearned_submissions subreddit

In [4]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/todayilearned_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'todayilearned')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("todayilearned_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 435
Lines: 900000 | Edges Found: 884
Lines: 1000000 | Edges Found: 1347
Lines: 1100000 | Edges Found: 1840
Lines: 1200000 | Edges Found: 2217
Lines: 1300000 | Edges Found: 2609
Lines: 1400000 | Edges Found: 2901
Lines: 1500000 | Edges Found: 3217
Lines: 1600000 | Edges Found: 3534
Lines: 1700000 | Edges Found: 3786
Lines: 1800000 | Edges Found: 4057
Lines: 1900000 | Edges Found: 4336
Lines: 2000000 | Edges Found: 4589
Lines: 2100000 | Edges Found: 4790
Lines: 2200000 | Edges Found: 5046
Complete! Extracted 5159 edges.


 # trees_submissions subreddit

In [5]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/trees_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'trees')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("trees_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 0
Lines: 600000 | Edges Found: 0
Lines: 700000 | Edges Found: 0
Lines: 800000 | Edges Found: 0
Lines: 900000 | Edges Found: 0
Lines: 1000000 | Edges Found: 0
Lines: 1100000 | Edges Found: 0
Lines: 1200000 | Edges Found: 0
Lines: 1300000 | Edges Found: 0
Lines: 1400000 | Edges Found: 0
Lines: 1500000 | Edges Found: 643
Lines: 1600000 | Edges Found: 2605
Lines: 1700000 | Edges Found: 4194
Lines: 1800000 | Edges Found: 5915
Lines: 1900000 | Edges Found: 7686
Lines: 2000000 | Edges Found: 9409
Lines: 2100000 | Edges Found: 11014
Lines: 2200000 | Edges Found: 12313
Lines: 2300000 | Edges Found: 13601
Lines: 2400000 | Edges Found: 15012
Lines: 2500000 | Edges Found: 16211
Lines: 2600000 | Edges Found: 17237
Lines: 2700000 | Edges Found: 18238
Lines: 2800000 | Edges Found: 19025
Lines: 2900000 | Edges Found: 19761
Lines: 3000000 | Edges Found

 # TrueOffMyChest_submissions subreddit

In [6]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/TrueOffMyChest_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'TrueOffMyChest')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("TrueOffMyChest_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 5378
Lines: 200000 | Edges Found: 9032
Lines: 300000 | Edges Found: 11097
Lines: 400000 | Edges Found: 12293
Lines: 500000 | Edges Found: 13272
Lines: 600000 | Edges Found: 13548
Lines: 700000 | Edges Found: 13789
Lines: 800000 | Edges Found: 14051
Lines: 900000 | Edges Found: 14565
Complete! Extracted 15286 edges.


 # Turkey_submissions subreddit

In [7]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Turkey_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Turkey')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Turkey_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2904
Lines: 200000 | Edges Found: 5903
Lines: 300000 | Edges Found: 8896
Complete! Extracted 11255 edges.


 # ufc_submissions subreddit

In [8]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/ufc_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ufc')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("ufc_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 649
Lines: 200000 | Edges Found: 985
Lines: 300000 | Edges Found: 1226
Lines: 400000 | Edges Found: 1542
Complete! Extracted 1658 edges.


 # Unity3D_submissions subreddit

In [9]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Unity3D_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Unity3D')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Unity3D_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3290
Lines: 200000 | Edges Found: 5620
Lines: 300000 | Edges Found: 7735
Complete! Extracted 8246 edges.


 # Vent_submissions subreddit

In [10]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Vent_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Vent')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Vent_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1968
Lines: 200000 | Edges Found: 2840
Lines: 300000 | Edges Found: 3002
Lines: 400000 | Edges Found: 3361
Lines: 500000 | Edges Found: 4086
Lines: 600000 | Edges Found: 4764
Complete! Extracted 5427 edges.


 # wallstreetbets_submissions subreddit

In [11]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/wallstreetbets_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'wallstreetbets')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("wallstreetbets_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2330
Lines: 200000 | Edges Found: 3872
Lines: 300000 | Edges Found: 5361
Lines: 400000 | Edges Found: 7411
Lines: 500000 | Edges Found: 8545
Lines: 600000 | Edges Found: 9756
Lines: 700000 | Edges Found: 10222
Lines: 800000 | Edges Found: 10521
Lines: 900000 | Edges Found: 10722
Lines: 1000000 | Edges Found: 11004
Lines: 1100000 | Edges Found: 11366
Lines: 1200000 | Edges Found: 11734
Lines: 1300000 | Edges Found: 12155
Lines: 1400000 | Edges Found: 12742
Lines: 1500000 | Edges Found: 13283
Lines: 1600000 | Edges Found: 13785
Lines: 1700000 | Edges Found: 14544
Lines: 1800000 | Edges Found: 15108
Lines: 1900000 | Edges Found: 15778
Lines: 2000000 | Edges Found: 16539
Lines: 2100000 | Edges Found: 17486
Lines: 2200000 | Edges Found: 18290
Lines: 2300000 | Edges Found: 18927
Lines: 2400000 | Edges Found: 19938
Lines: 2500000 | Edges Found: 21104
Complete! Extracted 21199 edges.


 # Warhammer_submissions subreddit

In [12]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Warhammer_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Warhammer')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Warhammer_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1680
Lines: 200000 | Edges Found: 2443
Complete! Extracted 2770 edges.


 # wow_submissions subreddit

In [13]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/wow_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'wow')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("wow_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 2269
Lines: 300000 | Edges Found: 5322
Lines: 400000 | Edges Found: 8006
Lines: 500000 | Edges Found: 9609
Lines: 600000 | Edges Found: 11520
Lines: 700000 | Edges Found: 12865
Lines: 800000 | Edges Found: 14050
Lines: 900000 | Edges Found: 15342
Lines: 1000000 | Edges Found: 17051
Lines: 1100000 | Edges Found: 18325
Lines: 1200000 | Edges Found: 19853
Lines: 1300000 | Edges Found: 21581
Lines: 1400000 | Edges Found: 23165
Complete! Extracted 23391 edges.


 # WritingPrompts_submissions subreddit

In [14]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/WritingPrompts_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'WritingPrompts')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("WritingPrompts_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3495
Lines: 200000 | Edges Found: 5951
Lines: 300000 | Edges Found: 7983
Lines: 400000 | Edges Found: 9494
Lines: 500000 | Edges Found: 10944
Lines: 600000 | Edges Found: 12195
Lines: 700000 | Edges Found: 13311
Lines: 800000 | Edges Found: 14210
Lines: 900000 | Edges Found: 15411
Lines: 1000000 | Edges Found: 17058
Complete! Extracted 18354 edges.


In [ ]:
 # xboxone_submissions subreddit

In [15]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/xboxone_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'xboxone')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("xboxone_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1276
Lines: 200000 | Edges Found: 2894
Lines: 300000 | Edges Found: 4095
Lines: 400000 | Edges Found: 5240
Lines: 500000 | Edges Found: 6266
Lines: 600000 | Edges Found: 7107
Lines: 700000 | Edges Found: 7804
Lines: 800000 | Edges Found: 8480
Lines: 900000 | Edges Found: 8934
Complete! Extracted 9072 edges.


 # chubby_submissions subreddit

In [1]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/chubby_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'chubby')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("chubby_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 613
Lines: 200000 | Edges Found: 669
Lines: 300000 | Edges Found: 790
Lines: 400000 | Edges Found: 822
Lines: 500000 | Edges Found: 839
Lines: 600000 | Edges Found: 851
Lines: 700000 | Edges Found: 866
Lines: 800000 | Edges Found: 882
Lines: 900000 | Edges Found: 889
Lines: 1000000 | Edges Found: 899
Lines: 1100000 | Edges Found: 915
Lines: 1200000 | Edges Found: 930
Lines: 1300000 | Edges Found: 960
Lines: 1400000 | Edges Found: 987
Lines: 1500000 | Edges Found: 1011
Lines: 1600000 | Edges Found: 1042
Lines: 1700000 | Edges Found: 1065
Lines: 1800000 | Edges Found: 1092
Lines: 1900000 | Edges Found: 1116
Lines: 2000000 | Edges Found: 1124
Lines: 2100000 | Edges Found: 1139
Lines: 2200000 | Edges Found: 1152
Lines: 2300000 | Edges Found: 1162
Lines: 2400000 | Edges Found: 1172
Complete! Extracted 1182 edges.


 # dirtypenpals_submissions subreddit

In [2]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/dirtypenpals_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'dirtypenpals')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("dirtypenpals_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 151
Lines: 200000 | Edges Found: 4303
Lines: 300000 | Edges Found: 10035
Lines: 400000 | Edges Found: 17130
Lines: 500000 | Edges Found: 23786
Lines: 600000 | Edges Found: 30329
Lines: 700000 | Edges Found: 35504
Lines: 800000 | Edges Found: 41607
Lines: 900000 | Edges Found: 48169
Lines: 1000000 | Edges Found: 54381
Lines: 1100000 | Edges Found: 59839
Lines: 1200000 | Edges Found: 65572
Lines: 1300000 | Edges Found: 71273
Lines: 1400000 | Edges Found: 77142
Lines: 1500000 | Edges Found: 83146
Lines: 1600000 | Edges Found: 88849
Lines: 1700000 | Edges Found: 94215
Lines: 1800000 | Edges Found: 99497
Lines: 1900000 | Edges Found: 105037
Lines: 2000000 | Edges Found: 110046
Lines: 2100000 | Edges Found: 115339
Lines: 2200000 | Edges Found: 120943
Lines: 2300000 | Edges Found: 126633
Lines: 2400000 | Edges Found: 132002
Lines: 2500000 | Edges Found: 137135
Lines: 2600000 | Edges Found: 142418
Lines: 2700000 | Edges Found: 147504
Lines: 2800000 | Edges Found: 1

 # dirtyr4r_submissions subreddit

In [3]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/dirtyr4r_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'dirtyr4r')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("dirtyr4r_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3135
Lines: 200000 | Edges Found: 4790
Lines: 300000 | Edges Found: 6060
Lines: 400000 | Edges Found: 7148
Lines: 500000 | Edges Found: 8349
Lines: 600000 | Edges Found: 9252
Lines: 700000 | Edges Found: 10299
Lines: 800000 | Edges Found: 11308
Lines: 900000 | Edges Found: 12359
Lines: 1000000 | Edges Found: 13411
Lines: 1100000 | Edges Found: 14247
Lines: 1200000 | Edges Found: 15170
Lines: 1300000 | Edges Found: 16184
Lines: 1400000 | Edges Found: 17013
Lines: 1500000 | Edges Found: 17791
Lines: 1600000 | Edges Found: 18643
Lines: 1700000 | Edges Found: 19422
Lines: 1800000 | Edges Found: 20315
Lines: 1900000 | Edges Found: 21183
Lines: 2000000 | Edges Found: 21997
Lines: 2100000 | Edges Found: 23046
Lines: 2200000 | Edges Found: 23960
Lines: 2300000 | Edges Found: 24759
Lines: 2400000 | Edges Found: 25225
Lines: 2500000 | Edges Found: 25753
Lines: 2600000 | Edges Found: 26315
Lines: 2700000 | Edges Found: 26880
Lines: 2800000 | Edges Found: 27398
Lines: 

 # gonewild_submissions subreddit

In [4]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/gonewild_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'gonewild')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("gonewild_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 0
Lines: 400000 | Edges Found: 0
Lines: 500000 | Edges Found: 282
Lines: 600000 | Edges Found: 892
Lines: 700000 | Edges Found: 1369
Lines: 800000 | Edges Found: 1858
Lines: 900000 | Edges Found: 2357
Lines: 1000000 | Edges Found: 2722
Lines: 1100000 | Edges Found: 3131
Lines: 1200000 | Edges Found: 3497
Lines: 1300000 | Edges Found: 3719
Lines: 1400000 | Edges Found: 3874
Lines: 1500000 | Edges Found: 3995
Lines: 1600000 | Edges Found: 4099
Lines: 1700000 | Edges Found: 4197
Lines: 1800000 | Edges Found: 4280
Lines: 1900000 | Edges Found: 4383
Lines: 2000000 | Edges Found: 4440
Lines: 2100000 | Edges Found: 4503
Lines: 2200000 | Edges Found: 4554
Lines: 2300000 | Edges Found: 4585
Lines: 2400000 | Edges Found: 4626
Lines: 2500000 | Edges Found: 4678
Lines: 2600000 | Edges Found: 4708
Lines: 2700000 | Edges Found: 4755
Lines: 2800000 | Edges Found: 4796
Lines: 2900000 | Edges Found: 4841
Lines: 3

 # 196_submissions subreddit

In [5]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/196_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', '196')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("196_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 671
Lines: 200000 | Edges Found: 895
Lines: 300000 | Edges Found: 1189
Lines: 400000 | Edges Found: 1518
Lines: 500000 | Edges Found: 1882
Lines: 600000 | Edges Found: 2307
Lines: 700000 | Edges Found: 2694
Lines: 800000 | Edges Found: 3567
Lines: 900000 | Edges Found: 3964
Lines: 1000000 | Edges Found: 4034
Lines: 1100000 | Edges Found: 4067
Lines: 1200000 | Edges Found: 4137
Lines: 1300000 | Edges Found: 4176
Lines: 1400000 | Edges Found: 4226
Lines: 1500000 | Edges Found: 4330
Lines: 1600000 | Edges Found: 4421
Complete! Extracted 4441 edges.


 # AmItheAsshole_submissions subreddit

In [6]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/AmItheAsshole_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'AmItheAsshole')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("AmItheAsshole_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2098
Lines: 200000 | Edges Found: 3407
Lines: 300000 | Edges Found: 4565
Lines: 400000 | Edges Found: 5625
Lines: 500000 | Edges Found: 6481
Lines: 600000 | Edges Found: 7364
Lines: 700000 | Edges Found: 8227
Lines: 800000 | Edges Found: 8994
Lines: 900000 | Edges Found: 9604
Lines: 1000000 | Edges Found: 10170
Lines: 1100000 | Edges Found: 10683
Lines: 1200000 | Edges Found: 11247
Lines: 1300000 | Edges Found: 11950
Lines: 1400000 | Edges Found: 12558
Lines: 1500000 | Edges Found: 13131
Lines: 1600000 | Edges Found: 13646
Lines: 1700000 | Edges Found: 14191
Lines: 1800000 | Edges Found: 14707
Lines: 1900000 | Edges Found: 15213
Lines: 2000000 | Edges Found: 16210
Lines: 2100000 | Edges Found: 17420
Lines: 2200000 | Edges Found: 18350
Lines: 2300000 | Edges Found: 19327
Lines: 2400000 | Edges Found: 20129
Lines: 2500000 | Edges Found: 20823
Complete! Extracted 20852 edges.


 # autotldr_submissions subreddit

In [8]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/autotldr_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'autotldr')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("autotldr_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 385240
Lines: 200000 | Edges Found: 1030304
Lines: 300000 | Edges Found: 1687940
Lines: 400000 | Edges Found: 2285539
Lines: 500000 | Edges Found: 2920844
Lines: 600000 | Edges Found: 3579214
Complete! Extracted 3935823 edges.


In [ ]:
 # DadWouldBeProud_submissions subreddit

In [9]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/DadWouldBeProud_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'DadWouldBeProud')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("DadWouldBeProud_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 87
Lines: 200000 | Edges Found: 97
Lines: 300000 | Edges Found: 99
Lines: 400000 | Edges Found: 104
Lines: 500000 | Edges Found: 112
Lines: 600000 | Edges Found: 127
Lines: 700000 | Edges Found: 139
Lines: 800000 | Edges Found: 155
Lines: 900000 | Edges Found: 160
Lines: 1000000 | Edges Found: 165
Lines: 1100000 | Edges Found: 167
Lines: 1200000 | Edges Found: 173
Lines: 1300000 | Edges Found: 180
Complete! Extracted 186 edges.


 # depression_submissions subreddit

In [10]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/depression_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'depression')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("depression_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 588
Lines: 200000 | Edges Found: 3967
Lines: 300000 | Edges Found: 5813
Lines: 400000 | Edges Found: 7197
Lines: 500000 | Edges Found: 8386
Lines: 600000 | Edges Found: 9387
Lines: 700000 | Edges Found: 10349
Lines: 800000 | Edges Found: 11334
Lines: 900000 | Edges Found: 12121
Lines: 1000000 | Edges Found: 12882
Lines: 1100000 | Edges Found: 13471
Lines: 1200000 | Edges Found: 14092
Lines: 1300000 | Edges Found: 14647
Lines: 1400000 | Edges Found: 15067
Lines: 1500000 | Edges Found: 15543
Lines: 1600000 | Edges Found: 16062
Lines: 1700000 | Edges Found: 16697
Complete! Extracted 17245 edges.


 # DnD_submissions subreddit

In [11]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/DnD_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'DnD')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("DnD_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 4898
Lines: 200000 | Edges Found: 9536
Lines: 300000 | Edges Found: 13341
Lines: 400000 | Edges Found: 16814
Lines: 500000 | Edges Found: 19862
Lines: 600000 | Edges Found: 22496
Lines: 700000 | Edges Found: 24782
Lines: 800000 | Edges Found: 27039
Lines: 900000 | Edges Found: 29055
Lines: 1000000 | Edges Found: 31284
Lines: 1100000 | Edges Found: 33462
Lines: 1200000 | Edges Found: 36107
Complete! Extracted 37501 edges.


 # Eldenring_submissions subreddit

In [12]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Eldenring_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Eldenring')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Eldenring_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 812
Lines: 200000 | Edges Found: 1603
Lines: 300000 | Edges Found: 2670
Lines: 400000 | Edges Found: 3712
Lines: 500000 | Edges Found: 5049
Lines: 600000 | Edges Found: 6106
Lines: 700000 | Edges Found: 7405
Lines: 800000 | Edges Found: 8901
Lines: 900000 | Edges Found: 10366
Lines: 1000000 | Edges Found: 12220
Lines: 1100000 | Edges Found: 13649
Lines: 1200000 | Edges Found: 14661
Complete! Extracted 15971 edges.


 # FortNiteBR_submissions subreddit

In [13]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/FortNiteBR_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'FortNiteBR')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("FortNiteBR_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 717
Lines: 200000 | Edges Found: 1163
Lines: 300000 | Edges Found: 1642
Lines: 400000 | Edges Found: 2149
Lines: 500000 | Edges Found: 2566
Lines: 600000 | Edges Found: 2988
Lines: 700000 | Edges Found: 3461
Lines: 800000 | Edges Found: 3969
Lines: 900000 | Edges Found: 4600
Lines: 1000000 | Edges Found: 5001
Lines: 1100000 | Edges Found: 5390
Lines: 1200000 | Edges Found: 5779
Lines: 1300000 | Edges Found: 6111
Lines: 1400000 | Edges Found: 6493
Lines: 1500000 | Edges Found: 7077
Lines: 1600000 | Edges Found: 7573
Lines: 1700000 | Edges Found: 7968
Lines: 1800000 | Edges Found: 8374
Lines: 1900000 | Edges Found: 8783
Lines: 2000000 | Edges Found: 9186
Lines: 2100000 | Edges Found: 9640
Lines: 2200000 | Edges Found: 9934
Lines: 2300000 | Edges Found: 10243
Lines: 2400000 | Edges Found: 10536
Lines: 2500000 | Edges Found: 10854
Lines: 2600000 | Edges Found: 11102
Lines: 2700000 | Edges Found: 11398
Lines: 2800000 | Edges Found: 11849
Lines: 2900000 | Edges F

 # NoFap_submissions subreddit

In [9]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/NoFap_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'NoFap')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("NoFap_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 1530
Lines: 300000 | Edges Found: 4009
Lines: 400000 | Edges Found: 5813
Lines: 500000 | Edges Found: 7141
Lines: 600000 | Edges Found: 8196
Lines: 700000 | Edges Found: 9180
Lines: 800000 | Edges Found: 10203
Lines: 900000 | Edges Found: 11118
Lines: 1000000 | Edges Found: 11987
Lines: 1100000 | Edges Found: 12743
Lines: 1200000 | Edges Found: 13394
Lines: 1300000 | Edges Found: 13947
Lines: 1400000 | Edges Found: 14508
Lines: 1500000 | Edges Found: 15063
Lines: 1600000 | Edges Found: 15497
Lines: 1700000 | Edges Found: 15956
Lines: 1800000 | Edges Found: 16374
Lines: 1900000 | Edges Found: 16805
Lines: 2000000 | Edges Found: 17294
Lines: 2100000 | Edges Found: 17845
Lines: 2200000 | Edges Found: 18325
Complete! Extracted 18377 edges.


 # nosleep_submissions subreddit

In [8]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/nosleep_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'nosleep')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("nosleep_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1710
Lines: 200000 | Edges Found: 4115
Lines: 300000 | Edges Found: 9210
Lines: 400000 | Edges Found: 19052
Complete! Extracted 20979 edges.


 # NoStupidQuestions_submissions subreddit

In [7]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/NoStupidQuestions_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'NoStupidQuestions')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("NoStupidQuestions_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3762
Lines: 200000 | Edges Found: 7089
Lines: 300000 | Edges Found: 10256
Lines: 400000 | Edges Found: 13007
Lines: 500000 | Edges Found: 15677
Lines: 600000 | Edges Found: 18533
Lines: 700000 | Edges Found: 21080
Lines: 800000 | Edges Found: 23766
Lines: 900000 | Edges Found: 26372
Lines: 1000000 | Edges Found: 28702
Lines: 1100000 | Edges Found: 30978
Lines: 1200000 | Edges Found: 33054
Lines: 1300000 | Edges Found: 35114
Lines: 1400000 | Edges Found: 37003
Lines: 1500000 | Edges Found: 39125
Lines: 1600000 | Edges Found: 41588
Lines: 1700000 | Edges Found: 43562
Lines: 1800000 | Edges Found: 45370
Lines: 1900000 | Edges Found: 47334
Lines: 2000000 | Edges Found: 49238
Lines: 2100000 | Edges Found: 50940
Lines: 2200000 | Edges Found: 52426
Lines: 2300000 | Edges Found: 53734
Lines: 2400000 | Edges Found: 55114
Lines: 2500000 | Edges Found: 56574
Lines: 2600000 | Edges Found: 57906
Lines: 2700000 | Edges Found: 59237
Lines: 2800000 | Edges Found: 60474
Lin

 # PokemonGoRaids_submissions subreddit

In [6]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/PokemonGoRaids_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'PokemonGoRaids')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("PokemonGoRaids_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 54
Lines: 200000 | Edges Found: 72
Lines: 300000 | Edges Found: 93
Lines: 400000 | Edges Found: 110
Lines: 500000 | Edges Found: 120
Lines: 600000 | Edges Found: 129
Lines: 700000 | Edges Found: 134
Lines: 800000 | Edges Found: 141
Lines: 900000 | Edges Found: 144
Lines: 1000000 | Edges Found: 146
Lines: 1100000 | Edges Found: 148
Lines: 1200000 | Edges Found: 161
Lines: 1300000 | Edges Found: 164
Lines: 1400000 | Edges Found: 169
Lines: 1500000 | Edges Found: 170
Lines: 1600000 | Edges Found: 173
Lines: 1700000 | Edges Found: 179
Lines: 1800000 | Edges Found: 183
Lines: 1900000 | Edges Found: 187
Lines: 2000000 | Edges Found: 190
Lines: 2100000 | Edges Found: 193
Lines: 2200000 | Edges Found: 201
Lines: 2300000 | Edges Found: 247
Lines: 2400000 | Edges Found: 268
Lines: 2500000 | Edges Found: 280
Lines: 2600000 | Edges Found: 302
Lines: 2700000 | Edges Found: 343
Lines: 2800000 | Edges Found: 374
Lines: 2900000 | Edges Found: 383
Lines: 3000000 | Edges Fou

 # r4r_submissions subreddit

In [5]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/r4r_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'r4r')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("r4r_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 2984
Lines: 300000 | Edges Found: 6235
Lines: 400000 | Edges Found: 7657
Lines: 500000 | Edges Found: 9096
Lines: 600000 | Edges Found: 10585
Lines: 700000 | Edges Found: 11718
Lines: 800000 | Edges Found: 12949
Lines: 900000 | Edges Found: 14143
Lines: 1000000 | Edges Found: 15092
Lines: 1100000 | Edges Found: 15938
Lines: 1200000 | Edges Found: 17064
Lines: 1300000 | Edges Found: 18398
Lines: 1400000 | Edges Found: 19063
Lines: 1500000 | Edges Found: 20094
Lines: 1600000 | Edges Found: 21220
Lines: 1700000 | Edges Found: 22777
Lines: 1800000 | Edges Found: 25085
Lines: 1900000 | Edges Found: 28593
Lines: 2000000 | Edges Found: 31431
Lines: 2100000 | Edges Found: 34087
Lines: 2200000 | Edges Found: 36815
Complete! Extracted 38652 edges.


 # Rainbow6_submissions subreddit

In [4]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Rainbow6_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Rainbow6')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Rainbow6_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1363
Lines: 200000 | Edges Found: 2478
Lines: 300000 | Edges Found: 3375
Lines: 400000 | Edges Found: 4296
Lines: 500000 | Edges Found: 5095
Lines: 600000 | Edges Found: 5946
Lines: 700000 | Edges Found: 6529
Lines: 800000 | Edges Found: 7180
Lines: 900000 | Edges Found: 7700
Lines: 1000000 | Edges Found: 8197
Lines: 1100000 | Edges Found: 8655
Lines: 1200000 | Edges Found: 9105
Lines: 1300000 | Edges Found: 9522
Lines: 1400000 | Edges Found: 9937
Lines: 1500000 | Edges Found: 10359
Lines: 1600000 | Edges Found: 10827
Complete! Extracted 11220 edges.


 # Repsneakers_submissions subreddit

In [3]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Repsneakers_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Repsneakers')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Repsneakers_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1050
Lines: 200000 | Edges Found: 1876
Lines: 300000 | Edges Found: 2315
Lines: 400000 | Edges Found: 2794
Lines: 500000 | Edges Found: 3205
Lines: 600000 | Edges Found: 3420
Lines: 700000 | Edges Found: 3579
Lines: 800000 | Edges Found: 3750
Lines: 900000 | Edges Found: 3954
Lines: 1000000 | Edges Found: 4259
Lines: 1100000 | Edges Found: 4653
Lines: 1200000 | Edges Found: 5116
Complete! Extracted 5242 edges.


 # RocketLeague_submissions subreddit

In [2]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/RocketLeague_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'RocketLeague')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("RocketLeague_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1406
Lines: 200000 | Edges Found: 2642
Lines: 300000 | Edges Found: 3902
Lines: 400000 | Edges Found: 5085
Lines: 500000 | Edges Found: 6038
Lines: 600000 | Edges Found: 6672
Lines: 700000 | Edges Found: 7161
Lines: 800000 | Edges Found: 7636
Lines: 900000 | Edges Found: 7956
Lines: 1000000 | Edges Found: 8338
Lines: 1100000 | Edges Found: 8887
Lines: 1200000 | Edges Found: 9445
Complete! Extracted 9993 edges.


 # ThickThighs_submissions subreddit

In [1]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/ThickThighs_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'ThickThighs')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("ThickThighs_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 241
Lines: 200000 | Edges Found: 323
Lines: 300000 | Edges Found: 340
Lines: 400000 | Edges Found: 374
Lines: 500000 | Edges Found: 405
Lines: 600000 | Edges Found: 442
Lines: 700000 | Edges Found: 447
Lines: 800000 | Edges Found: 451
Lines: 900000 | Edges Found: 456
Lines: 1000000 | Edges Found: 465
Lines: 1100000 | Edges Found: 471
Lines: 1200000 | Edges Found: 474
Complete! Extracted 474 edges.


 # amihot_submissions subreddit

In [10]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/amihot_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'amihot')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("amihot_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 195
Lines: 200000 | Edges Found: 223
Lines: 300000 | Edges Found: 251
Lines: 400000 | Edges Found: 285
Lines: 500000 | Edges Found: 302
Lines: 600000 | Edges Found: 322
Lines: 700000 | Edges Found: 365
Lines: 800000 | Edges Found: 392
Lines: 900000 | Edges Found: 424
Lines: 1000000 | Edges Found: 442
Lines: 1100000 | Edges Found: 453
Lines: 1200000 | Edges Found: 459
Lines: 1300000 | Edges Found: 467
Complete! Extracted 470 edges.


 # AnimalCrossing_submissions subreddit

In [11]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/AnimalCrossing_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'AnimalCrossing')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("AnimalCrossing_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1078
Lines: 200000 | Edges Found: 1515
Lines: 300000 | Edges Found: 1857
Lines: 400000 | Edges Found: 2212
Lines: 500000 | Edges Found: 2506
Lines: 600000 | Edges Found: 2864
Lines: 700000 | Edges Found: 3137
Lines: 800000 | Edges Found: 3381
Lines: 900000 | Edges Found: 3568
Lines: 1000000 | Edges Found: 3868
Lines: 1100000 | Edges Found: 4350
Complete! Extracted 4383 edges.


 # Animemes_submissions subreddit

In [12]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Animemes_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Animemes')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Animemes_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 497
Lines: 200000 | Edges Found: 1765
Lines: 300000 | Edges Found: 2290
Lines: 400000 | Edges Found: 2673
Lines: 500000 | Edges Found: 3019
Lines: 600000 | Edges Found: 3325
Lines: 700000 | Edges Found: 3635
Lines: 800000 | Edges Found: 3859
Lines: 900000 | Edges Found: 4198
Lines: 1000000 | Edges Found: 4549
Lines: 1100000 | Edges Found: 4736
Complete! Extracted 4950 edges.


 # ClashRoyale_submissionssubreddit

In [13]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/ClashRoyale_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Animemes')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Animemes_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1765
Lines: 200000 | Edges Found: 2944
Lines: 300000 | Edges Found: 4044
Lines: 400000 | Edges Found: 4761
Lines: 500000 | Edges Found: 5294
Lines: 600000 | Edges Found: 5808
Lines: 700000 | Edges Found: 6325
Lines: 800000 | Edges Found: 6635
Lines: 900000 | Edges Found: 6888
Lines: 1000000 | Edges Found: 7229
Complete! Extracted 7444 edges.


 # dirtykikpals_submissions subreddit

In [14]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/dirtykikpals_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'dirtykikpals')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("dirtykikpals_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 845
Lines: 200000 | Edges Found: 2072
Lines: 300000 | Edges Found: 2983
Lines: 400000 | Edges Found: 3752
Lines: 500000 | Edges Found: 4414
Lines: 600000 | Edges Found: 5124
Lines: 700000 | Edges Found: 6061
Lines: 800000 | Edges Found: 6993
Lines: 900000 | Edges Found: 7870
Lines: 1000000 | Edges Found: 8746
Lines: 1100000 | Edges Found: 9686
Lines: 1200000 | Edges Found: 10531
Lines: 1300000 | Edges Found: 11463
Lines: 1400000 | Edges Found: 12427
Lines: 1500000 | Edges Found: 13590
Lines: 1600000 | Edges Found: 14462
Lines: 1700000 | Edges Found: 15300
Lines: 1800000 | Edges Found: 16153
Lines: 1900000 | Edges Found: 16926
Lines: 2000000 | Edges Found: 17778
Lines: 2100000 | Edges Found: 18763
Lines: 2200000 | Edges Found: 19529
Lines: 2300000 | Edges Found: 20330
Lines: 2400000 | Edges Found: 21073
Lines: 2500000 | Edges Found: 21957
Lines: 2600000 | Edges Found: 22875
Lines: 2700000 | Edges Found: 23822
Lines: 2800000 | Edges Found: 25030
Lines: 290000

 # FantasticBreasts_submissions subreddit

In [15]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/FantasticBreasts_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'FantasticBreasts')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("FantasticBreasts_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 114
Lines: 200000 | Edges Found: 119
Lines: 300000 | Edges Found: 121
Lines: 400000 | Edges Found: 122
Lines: 500000 | Edges Found: 124
Lines: 600000 | Edges Found: 127
Complete! Extracted 128 edges.


 # Genshin_Impact_submissions subreddit

In [16]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Genshin_Impact_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Genshin_Impact')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Genshin_Impact_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 874
Lines: 200000 | Edges Found: 1817
Lines: 300000 | Edges Found: 2497
Lines: 400000 | Edges Found: 3264
Lines: 500000 | Edges Found: 3876
Lines: 600000 | Edges Found: 4469
Lines: 700000 | Edges Found: 5728
Lines: 800000 | Edges Found: 6563
Lines: 900000 | Edges Found: 7303
Lines: 1000000 | Edges Found: 8314
Lines: 1100000 | Edges Found: 9074
Complete! Extracted 9168 edges.


 # GlobalOffensive_submissions subreddit

In [17]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/GlobalOffensive_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'GlobalOffensive')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("GlobalOffensive_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1249
Lines: 200000 | Edges Found: 2890
Lines: 300000 | Edges Found: 4624
Lines: 400000 | Edges Found: 6069
Lines: 500000 | Edges Found: 7450
Lines: 600000 | Edges Found: 8577
Lines: 700000 | Edges Found: 9698
Lines: 800000 | Edges Found: 10892
Lines: 900000 | Edges Found: 12233
Lines: 1000000 | Edges Found: 13403
Lines: 1100000 | Edges Found: 15221
Lines: 1200000 | Edges Found: 17575
Lines: 1300000 | Edges Found: 19705
Lines: 1400000 | Edges Found: 21946
Lines: 1500000 | Edges Found: 24251
Lines: 1600000 | Edges Found: 26249
Lines: 1700000 | Edges Found: 28102
Lines: 1800000 | Edges Found: 30172
Lines: 1900000 | Edges Found: 33925
Lines: 2000000 | Edges Found: 93125
Complete! Extracted 115968 edges.


In [ ]:
 # hardwareswap_submissions subreddit

In [18]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/hardwareswap_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'hardwareswap')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("hardwareswap_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3620
Lines: 200000 | Edges Found: 7223
Lines: 300000 | Edges Found: 10993
Lines: 400000 | Edges Found: 15152
Lines: 500000 | Edges Found: 20639
Lines: 600000 | Edges Found: 26020
Lines: 700000 | Edges Found: 31131
Lines: 800000 | Edges Found: 35146
Lines: 900000 | Edges Found: 39787
Lines: 1000000 | Edges Found: 43861
Lines: 1100000 | Edges Found: 48869
Lines: 1200000 | Edges Found: 52776
Complete! Extracted 53714 edges.


 # IRLgirls_submissions subreddit

In [19]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/IRLgirls_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'IRLgirls')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("IRLgirls_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 86
Lines: 200000 | Edges Found: 113
Lines: 300000 | Edges Found: 140
Lines: 400000 | Edges Found: 155
Lines: 500000 | Edges Found: 169
Lines: 600000 | Edges Found: 174
Lines: 700000 | Edges Found: 193
Lines: 800000 | Edges Found: 234
Lines: 900000 | Edges Found: 240
Lines: 1000000 | Edges Found: 244
Lines: 1100000 | Edges Found: 268
Lines: 1200000 | Edges Found: 272
Lines: 1300000 | Edges Found: 277
Lines: 1400000 | Edges Found: 284
Lines: 1500000 | Edges Found: 287
Complete! Extracted 289 edges.


 # itookapicture_submissions subreddit

In [20]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/itookapicture_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'itookapicture')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("itookapicture_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 207
Lines: 300000 | Edges Found: 505
Lines: 400000 | Edges Found: 616
Lines: 500000 | Edges Found: 689
Lines: 600000 | Edges Found: 731
Lines: 700000 | Edges Found: 760
Lines: 800000 | Edges Found: 795
Lines: 900000 | Edges Found: 821
Lines: 1000000 | Edges Found: 854
Lines: 1100000 | Edges Found: 871
Complete! Extracted 883 edges.


 # KGBTR_submissions subreddit

In [21]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/KGBTR_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'KGBTR')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("KGBTR_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 5354
Lines: 200000 | Edges Found: 6656
Lines: 300000 | Edges Found: 8259
Lines: 400000 | Edges Found: 10164
Lines: 500000 | Edges Found: 10828
Lines: 600000 | Edges Found: 13059
Lines: 700000 | Edges Found: 20243
Lines: 800000 | Edges Found: 22871
Lines: 900000 | Edges Found: 27836
Lines: 1000000 | Edges Found: 28264
Lines: 1100000 | Edges Found: 29495
Lines: 1200000 | Edges Found: 30297
Lines: 1300000 | Edges Found: 32165
Complete! Extracted 35476 edges.


 # LegalTeens_submissions subreddit

In [22]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/LegalTeens_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'LegalTeens')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("LegalTeens_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 516
Lines: 200000 | Edges Found: 575
Lines: 300000 | Edges Found: 586
Lines: 400000 | Edges Found: 598
Lines: 500000 | Edges Found: 607
Lines: 600000 | Edges Found: 626
Lines: 700000 | Edges Found: 637
Lines: 800000 | Edges Found: 655
Lines: 900000 | Edges Found: 660
Lines: 1000000 | Edges Found: 662
Complete! Extracted 668 edges.


 # lego_submissions subreddit

In [23]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/lego_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'lego')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("lego_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1443
Lines: 200000 | Edges Found: 2208
Lines: 300000 | Edges Found: 2683
Lines: 400000 | Edges Found: 3096
Lines: 500000 | Edges Found: 3439
Lines: 600000 | Edges Found: 3824
Lines: 700000 | Edges Found: 4443
Complete! Extracted 4457 edges.


 # listentothis_submissions subreddit

In [24]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/listentothis_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'listentothis')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("listentothis_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 0
Lines: 300000 | Edges Found: 824
Lines: 400000 | Edges Found: 1582
Lines: 500000 | Edges Found: 2181
Lines: 600000 | Edges Found: 2726
Lines: 700000 | Edges Found: 3223
Lines: 800000 | Edges Found: 3616
Lines: 900000 | Edges Found: 4002
Lines: 1000000 | Edges Found: 4398
Lines: 1100000 | Edges Found: 4757
Lines: 1200000 | Edges Found: 5118
Complete! Extracted 5457 edges.


 # MechanicAdvice_submissions subreddit

In [25]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/MechanicAdvice_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'MechanicAdvice')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("MechanicAdvice_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3783
Lines: 200000 | Edges Found: 6528
Lines: 300000 | Edges Found: 8349
Lines: 400000 | Edges Found: 9942
Lines: 500000 | Edges Found: 11452
Lines: 600000 | Edges Found: 13232
Lines: 700000 | Edges Found: 15038
Complete! Extracted 16531 edges.


 # OnePiece_submissions_submissions subreddit

In [26]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/OnePiece_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'OnePiece')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("OnePiece_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2013
Lines: 200000 | Edges Found: 3986
Lines: 300000 | Edges Found: 5221
Lines: 400000 | Edges Found: 6116
Lines: 500000 | Edges Found: 7469
Lines: 600000 | Edges Found: 8370
Lines: 700000 | Edges Found: 9673
Lines: 800000 | Edges Found: 10619
Complete! Extracted 11019 edges.


 # PrequelMemes_submissions subreddit

In [27]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/PrequelMemes_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'PrequelMemes')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("PrequelMemes_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 3832
Lines: 200000 | Edges Found: 7224
Lines: 300000 | Edges Found: 10158
Lines: 400000 | Edges Found: 12806
Lines: 500000 | Edges Found: 13911
Lines: 600000 | Edges Found: 14885
Lines: 700000 | Edges Found: 15680
Lines: 800000 | Edges Found: 16734
Lines: 900000 | Edges Found: 17402
Lines: 1000000 | Edges Found: 18078
Lines: 1100000 | Edges Found: 18636
Complete! Extracted 19116 edges.


 # raisedbynarcissists_submissions subreddit

In [28]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/raisedbynarcissists_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'raisedbynarcissists')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("raisedbynarcissists_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 7142
Lines: 200000 | Edges Found: 13078
Lines: 300000 | Edges Found: 16080
Lines: 400000 | Edges Found: 18679
Lines: 500000 | Edges Found: 20685
Lines: 600000 | Edges Found: 22652
Lines: 700000 | Edges Found: 24924
Complete! Extracted 25511 edges.


 # RealCute_submissions subreddit

In [29]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/RealCute_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'RealCute')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("RealCute_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 5
Lines: 200000 | Edges Found: 5
Lines: 300000 | Edges Found: 6
Lines: 400000 | Edges Found: 9
Lines: 500000 | Edges Found: 27
Lines: 600000 | Edges Found: 38
Lines: 700000 | Edges Found: 51
Complete! Extracted 52 edges.


 # Roleplaybuddy2_submissions subreddit

In [30]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Roleplaybuddy2_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Roleplaybuddy2')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Roleplaybuddy2_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1258
Lines: 200000 | Edges Found: 2413
Lines: 300000 | Edges Found: 4209
Lines: 400000 | Edges Found: 6024
Lines: 500000 | Edges Found: 7805
Complete! Extracted 9443 edges.


 # rule34_submissions subreddit

In [31]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/rule34_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'rule34')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("rule34_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1317
Lines: 200000 | Edges Found: 2394
Lines: 300000 | Edges Found: 2649
Lines: 400000 | Edges Found: 2825
Lines: 500000 | Edges Found: 2941
Lines: 600000 | Edges Found: 3092
Lines: 700000 | Edges Found: 3249
Complete! Extracted 3264 edges.


 # selfie_submissions subreddit

In [32]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/selfie_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'selfie')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("selfie_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 114
Lines: 200000 | Edges Found: 221
Lines: 300000 | Edges Found: 305
Lines: 400000 | Edges Found: 359
Lines: 500000 | Edges Found: 402
Lines: 600000 | Edges Found: 442
Lines: 700000 | Edges Found: 488
Lines: 800000 | Edges Found: 533
Lines: 900000 | Edges Found: 576
Lines: 1000000 | Edges Found: 619
Lines: 1100000 | Edges Found: 650
Complete! Extracted 675 edges.


 # shitposting_submissions subreddit

In [33]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/shitposting_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'shitposting')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("shitposting_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 553
Lines: 200000 | Edges Found: 1000
Lines: 300000 | Edges Found: 1284
Lines: 400000 | Edges Found: 1559
Lines: 500000 | Edges Found: 2007
Lines: 600000 | Edges Found: 2716
Lines: 700000 | Edges Found: 3154
Lines: 800000 | Edges Found: 3603
Lines: 900000 | Edges Found: 4048
Lines: 1000000 | Edges Found: 4453
Lines: 1100000 | Edges Found: 4901
Lines: 1200000 | Edges Found: 5155
Complete! Extracted 5336 edges.


 # SuicideWatch_submissions subreddit

In [34]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/SuicideWatch_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'SuicideWatch')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("SuicideWatch_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2161
Lines: 200000 | Edges Found: 3839
Lines: 300000 | Edges Found: 4841
Lines: 400000 | Edges Found: 5711
Lines: 500000 | Edges Found: 6258
Lines: 600000 | Edges Found: 6691
Lines: 700000 | Edges Found: 7077
Lines: 800000 | Edges Found: 7456
Lines: 900000 | Edges Found: 7801
Lines: 1000000 | Edges Found: 8225
Lines: 1100000 | Edges Found: 8776
Lines: 1200000 | Edges Found: 9307
Complete! Extracted 9534 edges.


 # TooAfraidToAsk_submissions subreddit

In [35]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/TooAfraidToAsk_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'TooAfraidToAsk')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("TooAfraidToAsk_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 2955
Lines: 200000 | Edges Found: 5475
Lines: 300000 | Edges Found: 7861
Lines: 400000 | Edges Found: 9408
Lines: 500000 | Edges Found: 10738
Lines: 600000 | Edges Found: 11681
Lines: 700000 | Edges Found: 13075
Complete! Extracted 13356 edges.


 # traps_submissions subreddit

In [36]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/traps_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'traps')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Ttraps_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 280
Lines: 200000 | Edges Found: 352
Lines: 300000 | Edges Found: 385
Lines: 400000 | Edges Found: 413
Lines: 500000 | Edges Found: 432
Lines: 600000 | Edges Found: 447
Lines: 700000 | Edges Found: 476
Lines: 800000 | Edges Found: 498
Lines: 900000 | Edges Found: 522
Complete! Extracted 523 edges.


 # travel_submissions subreddit

In [37]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/travel_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'travel')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("travel_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 682
Lines: 300000 | Edges Found: 2360
Lines: 400000 | Edges Found: 3459
Lines: 500000 | Edges Found: 4343
Lines: 600000 | Edges Found: 5244
Lines: 700000 | Edges Found: 6105
Lines: 800000 | Edges Found: 6917
Lines: 900000 | Edges Found: 7650
Lines: 1000000 | Edges Found: 8806
Lines: 1100000 | Edges Found: 10133
Lines: 1200000 | Edges Found: 11476
Complete! Extracted 12754 edges.


 # travel_submissions subreddit

In [38]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/travel_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'travel')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("travel_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 0
Lines: 200000 | Edges Found: 682
Lines: 300000 | Edges Found: 2360
Lines: 400000 | Edges Found: 3459
Lines: 500000 | Edges Found: 4343
Lines: 600000 | Edges Found: 5244
Lines: 700000 | Edges Found: 6105
Lines: 800000 | Edges Found: 6917
Lines: 900000 | Edges Found: 7650
Lines: 1000000 | Edges Found: 8806
Lines: 1100000 | Edges Found: 10133
Lines: 1200000 | Edges Found: 11476
Complete! Extracted 12754 edges.


 # TrueOffMyChest_submissions subreddit

In [39]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/TrueOffMyChest_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'TrueOffMyChest')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("TrueOffMyChest_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 5378
Lines: 200000 | Edges Found: 9032
Lines: 300000 | Edges Found: 11097
Lines: 400000 | Edges Found: 12293
Lines: 500000 | Edges Found: 13272
Lines: 600000 | Edges Found: 13548
Lines: 700000 | Edges Found: 13789
Lines: 800000 | Edges Found: 14051
Lines: 900000 | Edges Found: 14565
Complete! Extracted 15286 edges.


 # twinks_submissions subreddit

In [40]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/twinks_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'twinks')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("twinks_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 360
Lines: 200000 | Edges Found: 419
Lines: 300000 | Edges Found: 469
Lines: 400000 | Edges Found: 486
Lines: 500000 | Edges Found: 505
Lines: 600000 | Edges Found: 540
Lines: 700000 | Edges Found: 597
Lines: 800000 | Edges Found: 619
Lines: 900000 | Edges Found: 659
Lines: 1000000 | Edges Found: 681
Complete! Extracted 682 edges.


 # Warthunder_submissions subreddit

In [41]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/Warthunder_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'Warthunder')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("Warthunder_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 1462
Lines: 200000 | Edges Found: 2507
Lines: 300000 | Edges Found: 3481
Lines: 400000 | Edges Found: 4232
Lines: 500000 | Edges Found: 4868
Lines: 600000 | Edges Found: 5535
Lines: 700000 | Edges Found: 6207
Lines: 800000 | Edges Found: 7001
Lines: 900000 | Edges Found: 7971
Complete! Extracted 8483 edges.


 # whatsthisbug_submissions subreddit

In [42]:
import zstandard as zstd
import json
import io
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # For Signed Network

analyzer = SentimentIntensityAnalyzer()
START_2014 = 1388534400 
END_2025 = 1767225599 

file_path = "subreddits24/whatsthisbug_submissions.zst"
output_edges = []
count = 0

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(fh) as reader:
        # Added errors='replace' to handle Reddit emojis/symbols
        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
        
        for line in text_stream:
            try:
                line = line.strip()
                if not line: continue
                
                post = json.loads(line)
                created_utc = int(post.get('created_utc', 0))
                
                if START_2014 <= created_utc <= END_2025:
                    source_sub = post.get('subreddit', 'whatsthisbug')
                    title = post.get('title', '')
                    body = post.get('selftext', '')
                    text_to_scan = f"{title} {body}"
                    
                    targets = re.findall(r'r/([A-Za-z0-9_]+)', text_to_scan)
                    
                    if targets:
                        # STANDOUT MOVE: Calculate Sentiment NOW
                        # This creates your "Signed" Network
                        sentiment_score = analyzer.polarity_scores(text_to_scan)['compound']
                        label = -1 if sentiment_score <= -0.05 else 1
                        
                        for target in targets:
                            if target.lower() != source_sub.lower():
                                output_edges.append({
                                    'SOURCE': source_sub,
                                    'TARGET': target,
                                    'TIMESTAMP': created_utc,
                                    'SENTIMENT': label # Added for your Signed Network
                                })
                
                count += 1
                if count % 100000 == 0:
                    print(f"Lines: {count} | Edges Found: {len(output_edges)}")
                    
            except Exception as e:
                continue

# Save as Parquet for even better space saving on Mac
df_edges = pd.DataFrame(output_edges)
df_edges.to_csv("whatsthisbug_edges_2014_2025.csv", index=False)
print(f"Complete! Extracted {len(output_edges)} edges.")


Lines: 100000 | Edges Found: 654
Lines: 200000 | Edges Found: 1107
Lines: 300000 | Edges Found: 1395
Lines: 400000 | Edges Found: 1732
Lines: 500000 | Edges Found: 1984
Lines: 600000 | Edges Found: 2218
Lines: 700000 | Edges Found: 2547
Lines: 800000 | Edges Found: 2970
Complete! Extracted 3247 edges.
